# GMIC development analysis and orientation review

Keep private. CPU only. Attach notebook 11 version 2 and the official RSNA competition. Reproduce fixed development thresholds/review counts; display all three prior fit-image orientation flags. No model training, threshold retuning, final inference, or clinical image interpretation. See the embedded analysis plan.

In [ ]:
from pathlib import Path
import sys, subprocess, json
PROJECT = Path('/kaggle/working/mammo_analysis_project')
PROJECT.mkdir(exist_ok=True)
target = PROJECT / 'mammo_reliability/__init__.py'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text('"""Research software. No clinical validation is implied."""\n\n__version__ = "0.1.0"\n')
target = PROJECT / 'mammo_reliability/audit.py'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text('"""Validate original RSNA metadata without silently changing the cohort."""\nfrom __future__ import annotations\n\nimport hashlib\nimport importlib.metadata\nimport json\nimport platform\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.model_selection import StratifiedKFold, train_test_split\n\nBREAST_KEY = ["patient_id", "laterality"]\nSTANDARD_VIEWS = ["CC", "MLO"]\nREQUIRED = ["site_id", "patient_id", "image_id", "laterality", "view", "cancer", "density"]\nOUTCOME_ONLY = ["cancer", "biopsy", "invasive", "BIRADS", "difficult_negative_case"]\n\n\ndef require(condition: bool, message: str) -> None:\n    """Unlike assert, this check is not disabled by Python\'s -O option."""\n    if not bool(condition):\n        raise ValueError(message)\n\n\ndef file_sha256(path: str | Path) -> str:\n    h = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for block in iter(lambda: handle.read(1024 * 1024), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef environment() -> dict:\n    packages = {}\n    for name in ["numpy", "pandas", "scipy", "scikit-learn", "Pillow", "pydicom",\n                 "torch", "torchvision", "pylibjpeg", "pylibjpeg-openjpeg"]:\n        try:\n            packages[name] = importlib.metadata.version(name)\n        except importlib.metadata.PackageNotFoundError:\n            packages[name] = None\n    return {"python": platform.python_version(), "platform": platform.platform(), "packages": packages}\n\n\ndef write_json(path: str | Path, payload: dict) -> None:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_suffix(path.suffix + ".tmp")\n    temp.write_text(json.dumps(payload, indent=2, allow_nan=False) + "\\n")\n    temp.replace(path)\n\n\ndef locate_dataset() -> Path:\n    candidates = [Path("/kaggle/input/competitions/rsna-breast-cancer-detection"),\n                  Path("/kaggle/input/rsna-breast-cancer-detection")]\n    for root in candidates:\n        if (root / "train.csv").is_file():\n            return root\n    raise FileNotFoundError("Attach the official RSNA competition dataset or provide its local root explicitly.")\n\n\ndef audit_metadata(images: pd.DataFrame) -> tuple[pd.DataFrame, dict, dict[str, pd.DataFrame]]:\n    require(len(images) > 0, "Metadata is empty.")\n    require(set(REQUIRED).issubset(images.columns), f"Missing columns: {set(REQUIRED) - set(images.columns)}")\n    for col in ["site_id", "patient_id", "image_id", "laterality", "view"]:\n        require(images[col].notna().all(), f"Missing identifying information in {col}.")\n    for col in ["site_id", "patient_id", "image_id"]:\n        values = pd.to_numeric(images[col], errors="coerce")\n        require((values.notna() & np.isfinite(values) & (values >= 0) & (values == np.floor(values))).all(),\n                f"{col} must contain nonnegative integer identifiers.")\n    require(images["image_id"].is_unique, "Repeated image IDs require investigation.")\n    require(images["laterality"].isin(["L", "R"]).all(), "Unexpected laterality.")\n    require(images["cancer"].isin([0, 1]).all(), "Missing or nonbinary cancer labels.")\n    require((images.groupby("patient_id")["site_id"].nunique() == 1).all(), "A patient occurs at multiple sites.")\n    grouped = images.groupby(BREAST_KEY, sort=True)\n    require((grouped["cancer"].nunique(dropna=False) == 1).all(), "Cancer labels conflict within a breast.")\n    observed_density = images["density"].dropna()\n    require(observed_density.isin(list("ABCD")).all(), "Unexpected density category; inspect the source values.")\n\n    breasts = grouped.agg(site_id=("site_id", "first"), cancer=("cancer", "first"),\n                          n_images=("image_id", "size"), density=("density", "first"),\n                          n_density_labels=("density", "count"),\n                          n_density_values=("density", "nunique"))\n    views = pd.crosstab(index=[images[c] for c in BREAST_KEY], columns=images["view"])\n    views = views.reindex(index=breasts.index, columns=STANDARD_VIEWS, fill_value=0)\n    breasts["n_cc"] = views["CC"]\n    breasts["n_mlo"] = views["MLO"]\n    breasts["has_both_views"] = (breasts["n_cc"] > 0) & (breasts["n_mlo"] > 0)\n    breasts["density_missing"] = breasts["n_density_labels"] == 0\n    breasts["density_partly_missing"] = ((breasts["n_density_labels"] > 0) &\n                                           (breasts["n_density_labels"] < breasts["n_images"]))\n    breasts["density_conflict"] = breasts["n_density_values"] > 1\n    # The first available value is retained only if all available values agree.\n    breasts.loc[breasts["density_conflict"], "density"] = np.nan\n    breasts["density_group"] = breasts["density"].map({"A": "A/B", "B": "A/B", "C": "C/D", "D": "C/D"})\n    breasts["has_extra_images"] = breasts["n_images"] > 2\n    if "age" in images:\n        breasts["age"] = grouped["age"].first()\n        breasts["age_conflict"] = grouped["age"].nunique() > 1\n        breasts.loc[breasts["age_conflict"], "age"] = np.nan\n    breasts = breasts.reset_index()\n\n    extra = breasts.groupby(["site_id", "cancer"], as_index=False).agg(\n        breast_count=("cancer", "size"), breasts_with_extra_images=("has_extra_images", "sum"),\n        extra_image_fraction=("has_extra_images", "mean"))\n    density = breasts.groupby("site_id", as_index=False).agg(\n        breast_count=("cancer", "size"), all_missing=("density_missing", "sum"),\n        partly_missing=("density_partly_missing", "sum"), conflicts=("density_conflict", "sum"))\n    tables = {\n        "site_cancer": breasts.groupby(["site_id", "cancer"]).size().rename("breast_count").reset_index(),\n        "extra_images": extra, "density_availability": density,\n        "density_cancer": breasts.assign(density=breasts["density"].fillna("Missing/conflicting"))\n            .groupby(["site_id", "density", "cancer"]).size().rename("breast_count").reset_index(),\n        "image_views": images["view"].value_counts(dropna=False).rename_axis("view").rename("image_count").reset_index(),\n        "missing_columns": images.isna().sum().rename_axis("column").rename("missing_image_rows").reset_index(),\n    }\n    summary = {\n        "source": "computed_from_supplied_metadata", "image_rows": int(len(images)),\n        "patients": int(images["patient_id"].nunique()), "breasts": int(len(breasts)),\n        "positive_breasts": int(breasts["cancer"].sum()),\n        "positive_patients": int(breasts.groupby("patient_id")["cancer"].max().sum()),\n        "positive_breast_fraction": float(breasts["cancer"].mean()),\n        "breasts_missing_standard_view": int((~breasts["has_both_views"]).sum()),\n        "breasts_with_extra_images": int(breasts["has_extra_images"].sum()),\n        "density_conflicts": int(breasts["density_conflict"].sum()),\n        "pixel_duplicates_checked": False,\n    }\n    return breasts, summary, tables\n\n\ndef make_patient_folds(breasts: pd.DataFrame, n_splits: int = 5, seed: int = 20260909) -> pd.DataFrame:\n    require(breasts[BREAST_KEY].duplicated().sum() == 0, "Expected one row per breast.")\n    require(breasts["has_both_views"].all(), "Resolve breasts missing CC or MLO before defining the paired cohort.")\n    patients = breasts.groupby("patient_id", as_index=False).agg(site_id=("site_id", "first"),\n                                                               patient_cancer=("cancer", "max"))\n    strata = patients["site_id"].astype(str) + ":" + patients["patient_cancer"].astype(str)\n    require(strata.value_counts().min() >= n_splits, "Insufficient patients per site/cancer stratum for these folds.")\n    patients["outer_fold"] = -1\n    for fold, (_, test_idx) in enumerate(StratifiedKFold(n_splits, shuffle=True, random_state=seed).split(patients, strata)):\n        patients.loc[test_idx, "outer_fold"] = fold\n    require((patients["outer_fold"] >= 0).all(), "Some patients lack a fold.")\n    return patients\n\n\ndef partition_outer_fold(patients: pd.DataFrame, fold: int, seed: int = 20260909) -> pd.DataFrame:\n    require(patients["patient_id"].is_unique, "Split input must contain unique patients.")\n    require(fold in patients["outer_fold"].unique(), "Unknown outer fold.")\n    test = patients["outer_fold"] == fold\n    dev = patients.loc[~test].copy()\n    strata = dev["site_id"].astype(str) + ":" + dev["patient_cancer"].astype(str)\n    # Fractions of the complete cohort: fit 60%, tune 10%, calibrate 10%, test 20%.\n    fit_idx, held_idx = train_test_split(dev.index, test_size=0.25, stratify=strata, random_state=seed + fold)\n    held = dev.loc[held_idx]\n    held_strata = held["site_id"].astype(str) + ":" + held["patient_cancer"].astype(str)\n    tune_idx, cal_idx = train_test_split(held.index, test_size=0.5, stratify=held_strata, random_state=seed + 100 + fold)\n    result = patients.copy()\n    result["role"] = "test"\n    result.loc[fit_idx, "role"] = "fit"\n    result.loc[tune_idx, "role"] = "tune"\n    result.loc[cal_idx, "role"] = "calibrate"\n    require((result.loc[test, "role"] == "test").all(), "Outer test patients leaked into development.")\n    require((result.loc[~test, "role"] != "test").all(), "Development patient role was not assigned.")\n    for role, frame in result.groupby("role"):\n        require(set(frame["patient_cancer"].unique()) == {0, 1}, f"Role {role} lacks a class.")\n    return result\n')
target = PROJECT / 'mammo_reliability/evaluation.py'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text('"""Calibration and review-budget evaluation, with patient-cluster uncertainty."""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nimport hashlib\nimport math\n\nimport numpy as np\nimport pandas as pd\nfrom scipy.optimize import minimize\nfrom scipy.special import expit, logit\nfrom sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score\n\nfrom .audit import BREAST_KEY, require\n\n\ndef checked_probability(p) -> np.ndarray:\n    p = np.asarray(p, dtype=float)\n    require(p.ndim == 1 and len(p) > 0, "Expected a nonempty probability vector.")\n    require(np.isfinite(p).all() and ((p >= 0) & (p <= 1)).all(), "Invalid probabilities.")\n    return p\n\n\ndef checked_labels(y, length: int) -> np.ndarray:\n    y = np.asarray(y)\n    require(y.ndim == 1 and len(y) == length and np.isin(y, [0, 1]).all(), "Expected matching binary labels.")\n    return y.astype(int)\n\n\n@dataclass\nclass MonotonePlatt:\n    """Positive-slope logistic calibration; fit only on calibration patients."""\n    slope: float = 1.0\n    intercept: float = 0.0\n\n    def fit(self, p, y):\n        p = checked_probability(p)\n        y = checked_labels(y, len(p))\n        require(len(np.unique(y)) == 2, "Calibration needs both classes.")\n        z = logit(np.clip(p, 1e-7, 1 - 1e-7))\n\n        def objective(params):\n            q = np.exp(params[0]) * z + params[1]\n            return float(np.mean(np.logaddexp(0, q) - y * q) + 1e-6 * np.sum(np.square(params)))\n\n        fitted = minimize(objective, [0.0, 0.0], method="L-BFGS-B", bounds=[(-4, 4), (-20, 20)])\n        require(fitted.success and np.isfinite(fitted.fun), f"Calibration did not converge: {fitted.message}")\n        self.slope, self.intercept = float(np.exp(fitted.x[0])), float(fitted.x[1])\n        return self\n\n    def predict(self, p):\n        p = checked_probability(p)\n        return expit(self.slope * logit(np.clip(p, 1e-7, 1 - 1e-7)) + self.intercept)\n\n\ndef sensitivity_threshold(p, y, target: float = 0.90) -> float:\n    p = checked_probability(p)\n    y = checked_labels(y, len(p))\n    require(0 < target <= 1, "Sensitivity target must be in (0, 1].")\n    positives = np.sort(p[y == 1])[::-1]\n    require(len(positives) > 0, "No positive calibration breasts.")\n    # Prediction is positive for p >= threshold. Tied scores are all positive.\n    return float(positives[math.ceil(target * len(positives)) - 1])\n\n\ndef classification_metrics(y, p, threshold: float) -> dict:\n    p = checked_probability(p)\n    y = checked_labels(y, len(p))\n    require(np.isfinite(threshold) and 0 <= threshold <= 1, "Invalid fixed threshold.")\n    positive = p >= threshold\n    tp, fn = int(((y == 1) & positive).sum()), int(((y == 1) & ~positive).sum())\n    tn, fp = int(((y == 0) & ~positive).sum()), int(((y == 0) & positive).sum())\n    safe_ratio = lambda a, b: float(a / b) if b else None\n    bins = np.minimum((p * 10).astype(int), 9)\n    ece = sum(float((bins == b).mean() * abs(p[bins == b].mean() - y[bins == b].mean()))\n              for b in range(10) if (bins == b).any())\n    both = len(np.unique(y)) == 2\n    return {"n_breasts": len(y), "n_positive": int(y.sum()), "tp": tp, "fn": fn, "tn": tn, "fp": fp,\n            "sensitivity": safe_ratio(tp, tp + fn), "specificity": safe_ratio(tn, tn + fp),\n            "ppv_in_study_sample": safe_ratio(tp, tp + fp), "npv_in_study_sample": safe_ratio(tn, tn + fn),\n            "roc_auc": float(roc_auc_score(y, p)) if both else None,\n            "average_precision": float(average_precision_score(y, p)) if y.sum() else None,\n            "brier": float(brier_score_loss(y, p)), "log_loss": float(log_loss(y, p, labels=[0, 1])),\n            "ece_10_equal_width_bins": ece, "threshold": float(threshold)}\n\n\ndef ensemble_features(predictions: pd.DataFrame, expected_seeds: list[int]) -> pd.DataFrame:\n    keys = BREAST_KEY + ["seed"]\n    required = set(keys + ["z_cc", "z_mlo"])\n    require(required.issubset(predictions), "Prediction schema is incomplete.")\n    require(not predictions[keys].duplicated().any(), "Duplicate breast/member predictions.")\n    require(len(expected_seeds) >= 2 and len(set(expected_seeds)) == len(expected_seeds), "Ensemble needs distinct members.")\n    require(set(predictions["seed"].unique()) == set(expected_seeds), "Unexpected or missing ensemble seed.")\n    require((predictions.groupby(BREAST_KEY)["seed"].nunique() == len(expected_seeds)).all(), "Incomplete ensemble for a breast.")\n    require(np.isfinite(predictions[["z_cc", "z_mlo"]].to_numpy()).all(), "Nonfinite model predictions.")\n    tmp = predictions.copy()\n    tmp["p_member"] = expit((tmp["z_cc"] + tmp["z_mlo"]) / 2)\n    tmp["p_cc"] = expit(tmp["z_cc"])\n    tmp["p_mlo"] = expit(tmp["z_mlo"])\n    group = tmp.groupby(BREAST_KEY)\n    result = group.agg(p_raw=("p_member", "mean"), p_cc=("p_cc", "mean"), p_mlo=("p_mlo", "mean"))\n    result["ensemble_variance"] = group["p_member"].var(ddof=0)\n    result["view_disagreement"] = abs(result["p_cc"] - result["p_mlo"])\n    return result.reset_index()\n\n\ndef reference_percentile(reference, values) -> np.ndarray:\n    """Midrank empirical CDF learned from calibration negatives only."""\n    ref = np.sort(np.asarray(reference, dtype=float))\n    values = np.asarray(values, dtype=float)\n    require(len(ref) > 0 and np.isfinite(ref).all() and np.isfinite(values).all(), "Invalid percentile reference.")\n    return (np.searchsorted(ref, values, side="left") + np.searchsorted(ref, values, side="right")) / (2 * len(ref))\n\n\ndef add_review_scores(calibration: pd.DataFrame, target: pd.DataFrame, threshold: float) -> pd.DataFrame:\n    reference = calibration.loc[calibration["p_calibrated"] < threshold]\n    require(len(reference) > 0, "No calibration negatives for score normalization.")\n    result = target.copy()\n    result["score_confidence"] = result["p_calibrated"]\n    result["score_disagreement"] = result["view_disagreement"]\n    result["score_uncertainty"] = result["ensemble_variance"]\n    normalized = []\n    for col in ["p_calibrated", "view_disagreement", "ensemble_variance"]:\n        normalized.append(reference_percentile(reference[col], result[col]))\n    result["score_confidence_disagreement"] = (normalized[0] + normalized[1]) / 2\n    result["score_confidence_uncertainty"] = (normalized[0] + normalized[2]) / 2\n    result["score_combined"] = sum(normalized) / 3\n    result["predicted_positive"] = result["p_calibrated"] >= threshold\n    return result\n\n\ndef review_selection(frame: pd.DataFrame, score_column: str, budget: float = 0.05, seed: int = 20260909) -> pd.Series:\n    require(0 <= budget <= 1, "Review budget must be between zero and one.")\n    require(frame.index.is_unique, "Reset the dataframe index before selecting review candidates.")\n    require(not frame[BREAST_KEY].duplicated().any(), "Review input must contain unique breasts.")\n    require(frame["predicted_positive"].isin([True, False]).all(), "Missing decision flags.")\n    checked_labels(frame["cancer"], len(frame))\n    require(np.isfinite(frame[score_column]).all(), "Nonfinite review scores.")\n    negative = frame.loc[~frame["predicted_positive"].astype(bool)].copy()\n    # Tie ordering uses only identifiers and a fixed seed, never labels or table order.\n    negative["tie"] = [hashlib.sha256(f"{seed}:{p}:{l}".encode()).hexdigest()\n                       for p, l in negative[BREAST_KEY].itertuples(index=False, name=None)]\n    negative = negative.sort_values([score_column, "tie"], ascending=[False, True])\n    k = math.floor(budget * len(negative))\n    selection = pd.Series(False, index=frame.index)\n    selection.loc[negative.iloc[:k].index] = True\n    return selection\n\n\ndef review_at_budget(frame: pd.DataFrame, score_column: str, budget: float = 0.05, seed: int = 20260909) -> dict:\n    selected = review_selection(frame, score_column, budget, seed)\n    negative = frame.loc[~frame["predicted_positive"].astype(bool)]\n    k = int(selected.sum())\n    fn = int(negative["cancer"].sum())\n    caught = int(frame.loc[selected, "cancer"].sum())\n    return {"method": score_column, "budget": budget, "negative_breasts": len(negative),\n            "reviewed_breasts": k, "false_negatives": fn, "false_negatives_flagged": caught,\n            "fn_capture_fraction": float(caught / fn) if fn else None,\n            "review_yield": float(caught / k) if k else None,\n            "actual_review_fraction": float(k / len(negative)) if len(negative) else 0.0,\n            "random_expected_fn_capture": float(k / len(negative)) if len(negative) and fn else None,\n            "interpretation": "flagging potential review candidates; no human review or cancer recovery was measured"}\n\n\ndef paired_cluster_bootstrap(frame: pd.DataFrame, method: str = "score_combined", baseline: str = "score_confidence",\n                             budget: float = 0.05, repeats: int = 2000, seed: int = 20260909) -> dict:\n    """Patient bootstrap of pooled capture difference, allocating budget per fold.\n\n    Multiplicity weights represent sampled patient clusters, including both breasts.\n    All fitted scores/thresholds stay fixed. A replicate with zero total false\n    negatives is undefined. This does not include model-retraining uncertainty.\n    """\n    require(repeats > 0, "Bootstrap repetitions must be positive.")\n    require(0 <= budget <= 1, "Review budget must be between zero and one.")\n    require(not frame[BREAST_KEY].duplicated().any(), "Expected unique held-out breasts.")\n    require(frame.groupby("patient_id")["outer_fold"].nunique().max() == 1, "Patient spans held-out folds.")\n    rng = np.random.default_rng(seed)\n    fold_groups = []\n    for _, fold in frame.groupby("outer_fold"):\n        # Validation and point-estimate routines use the same fixed tie ordering.\n        review_selection(fold, method, budget, seed)\n        review_selection(fold, baseline, budget, seed)\n        patients = np.sort(fold["patient_id"].unique())\n        negative = fold.loc[~fold["predicted_positive"].astype(bool)]\n        group_idx = np.searchsorted(patients, negative["patient_id"].to_numpy())\n        labels = negative["cancer"].to_numpy(dtype=int)\n        ties = np.array([hashlib.sha256(f"{seed}:{p}:{l}".encode()).hexdigest()\n                         for p, l in negative[BREAST_KEY].itertuples(index=False, name=None)])\n        orders = [np.lexsort((ties, -negative[col].to_numpy(dtype=float))) for col in [method, baseline]]\n        fold_groups.append((len(patients), group_idx, labels, orders))\n    estimates, undefined = [], 0\n    for _ in range(repeats):\n        total_fn, total_difference = 0, 0\n        for n_patients, group_idx, labels, orders in fold_groups:\n            multiplicity = rng.multinomial(n_patients, np.full(n_patients, 1 / n_patients))\n            weight = multiplicity[group_idx]\n            total_fn += int(weight @ labels)\n            k = math.floor(budget * weight.sum())\n            caught = []\n            for order in orders:\n                ordered_weight = weight[order]\n                preceding = np.cumsum(ordered_weight) - ordered_weight\n                selected_weight = np.minimum(ordered_weight, np.maximum(0, k - preceding))\n                caught.append(int(selected_weight @ labels[order]))\n            total_difference += caught[0] - caught[1]\n        if total_fn == 0:\n            undefined += 1\n        else:\n            estimates.append(float(total_difference / total_fn))\n    point_difference, point_fn = 0, 0\n    for _, fold in frame.groupby("outer_fold"):\n        a = review_at_budget(fold, method, budget, seed)\n        b = review_at_budget(fold, baseline, budget, seed)\n        point_difference += a["false_negatives_flagged"] - b["false_negatives_flagged"]\n        point_fn += a["false_negatives"]\n    result = {"method": method, "baseline": baseline, "budget": budget,\n              "estimand": "pooled FN capture difference with the review budget applied separately in each held-out fold",\n              "point_estimate": float(point_difference / point_fn) if point_fn else None,\n              "valid_bootstrap_replicates": len(estimates), "undefined_replicates": undefined,\n              "requested_replicates": repeats, "ci95": None,\n              "uncertainty_scope": "conditional on fitted models, calibration, and threshold; not model retraining"}\n    if len(estimates) >= max(20, math.ceil(repeats * 0.8)) and result["point_estimate"] is not None:\n        result["ci95"] = [float(x) for x in np.quantile(estimates, [0.025, 0.975])]\n    return result\n')
target = PROJECT / 'mammo_reliability/imaging.py'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text('"""DICOM presentation processing, preserving the full field of view."""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport shutil\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nimport pydicom\nfrom pydicom.pixels import apply_modality_lut, apply_voi_lut\n\nfrom .audit import STANDARD_VIEWS, file_sha256, require, write_json\n\n\ndef normalize_pixels(raw: np.ndarray, presented: np.ndarray, photometric: str,\n                     padding_value=None, padding_limit=None) -> np.ndarray:\n    require(raw.ndim == 2 and presented.shape == raw.shape, "Only two-dimensional grayscale images are supported.")\n    require(photometric in ["MONOCHROME1", "MONOCHROME2"], "Unsupported photometric interpretation.")\n    valid = np.isfinite(presented)\n    if padding_value is not None:\n        if padding_limit is None:\n            valid &= raw != padding_value\n        else:\n            lower, upper = sorted([padding_value, padding_limit])\n            valid &= ~((raw >= lower) & (raw <= upper))\n    require(valid.any(), "No valid image pixels remain.")\n    low, high = float(presented[valid].min()), float(presented[valid].max())\n    require(high > low, "Uniform image after presentation processing.")\n    result = ((presented.astype(np.float32) - low) / (high - low)).clip(0, 1)\n    if photometric == "MONOCHROME1":\n        result = 1 - result\n    result[~valid] = 0\n    return result.astype(np.float32)\n\n\ndef letterbox(array: np.ndarray, height: int = 1024, width: int = 768) -> np.ndarray:\n    require(array.ndim == 2 and min(height, width) > 0, "Invalid image or target dimensions.")\n    h, w = array.shape\n    factor = min(height / h, width / w)\n    new_h, new_w = max(1, round(h * factor)), max(1, round(w * factor))\n    resized = np.asarray(Image.fromarray(array.astype(np.float32)).resize(\n        (new_w, new_h), resample=Image.Resampling.BILINEAR))\n    canvas = np.zeros((height, width), dtype=np.float32)\n    y, x = (height - new_h) // 2, (width - new_w) // 2\n    canvas[y:y + new_h, x:x + new_w] = resized\n    return canvas\n\n\ndef decode_dicom(path: str | Path, height: int = 1024, width: int = 768) -> tuple[np.ndarray, dict]:\n    ds = pydicom.dcmread(path)\n    raw = ds.pixel_array\n    if raw.ndim == 3 and raw.shape[0] == 1:\n        raw = raw[0]\n    require(raw.ndim == 2, "Multiframe/color DICOM requires a separate preprocessing decision.")\n    require("PresentationLUTSequence" not in ds, "Presentation LUT sequence requires image-specific review.")\n    photo = str(ds.get("PhotometricInterpretation", ""))\n    presented = apply_voi_lut(apply_modality_lut(raw, ds), ds)\n    normalized = normalize_pixels(raw, presented, photo, ds.get("PixelPaddingValue"), ds.get("PixelPaddingRangeLimit"))\n    digest = hashlib.sha256()\n    digest.update(str((raw.shape, str(raw.dtype))).encode())\n    digest.update(np.ascontiguousarray(raw).tobytes())\n    info = {"height_original": int(raw.shape[0]), "width_original": int(raw.shape[1]),\n            "photometric": photo, "decoded_pixel_sha256": digest.hexdigest(),\n            "voi_present": bool("VOILUTSequence" in ds or "WindowCenter" in ds),\n            "nonzero_fraction_after_normalization": float((normalized > 0).mean())}\n    return letterbox(normalized, height, width), info\n\n\ndef check_cross_patient_duplicates(manifest: pd.DataFrame) -> None:\n    require("decoded_pixel_sha256" in manifest, "Pixel duplicate audit was not performed.")\n    shared = manifest.groupby("decoded_pixel_sha256")["patient_id"].nunique()\n    require(not (shared > 1).any(), "Identical decoded images occur under different patients. Investigate/group them before training.")\n\n\ndef prepare_images(images: pd.DataFrame, data_root: str | Path, output: str | Path,\n                   height: int = 1024, width: int = 768, cache_images: bool = True) -> pd.DataFrame:\n    """Resume verified 16-bit PNG conversion, or audit pixels without caching.\n\n    Never skip failed images. A stopped run can resume from the last manifest.\n    An explicit disk check after the first 32 images prevents filling the volume.\n    """\n    data_root, output = Path(data_root), Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    source_hash = file_sha256(data_root / "train.csv")\n    config = {"version": 1, "height": height, "width": width, "cache_images": cache_images, "metadata_sha256": source_hash}\n    config_file, manifest_file = output / "preprocessing.json", output / "manifest.csv"\n    if config_file.exists():\n        require(json.loads(config_file.read_text()) == config, "Preprocessing settings/source changed; use a new output directory.")\n    else:\n        write_json(config_file, config)\n    rows = images.loc[images["view"].isin(STANDARD_VIEWS)].sort_values(["patient_id", "image_id"])\n    previous = pd.read_csv(manifest_file).to_dict("records") if manifest_file.exists() else []\n    recorded = {int(r["image_id"]): r for r in previous}\n    require(len(recorded) == len(previous), "Repeated image in preprocessing manifest.")\n    require(set(recorded).issubset(set(rows["image_id"].astype(int))), "Manifest has unexpected images.")\n    cached_sizes = []\n    for row in rows.itertuples(index=False):\n        image_id, patient_id = int(row.image_id), int(row.patient_id)\n        if image_id in recorded:\n            old = recorded[image_id]\n            if cache_images:\n                require(Path(old["cache_path"]).is_file(), "A recorded cached image is missing.")\n                require(file_sha256(old["cache_path"]) == old["cache_sha256"], "Cached image hash mismatch.")\n                cached_sizes.append(Path(old["cache_path"]).stat().st_size)\n            continue\n        source = data_root / "train_images" / str(patient_id) / f"{image_id}.dcm"\n        array, info = decode_dicom(source, height, width)\n        record = {"patient_id": patient_id, "image_id": image_id, "laterality": row.laterality,\n                  "view": row.view, "source_path": str(source), **info}\n        if cache_images:\n            cache_path = output / "images" / str(patient_id) / f"{image_id}.png"\n            cache_path.parent.mkdir(parents=True, exist_ok=True)\n            Image.fromarray(np.round(array * 65535).astype(np.uint16)).save(cache_path)\n            record.update(cache_path=str(cache_path), cache_sha256=file_sha256(cache_path))\n            cached_sizes.append(cache_path.stat().st_size)\n        recorded[image_id] = record\n        if len(recorded) % 32 == 0 or len(recorded) == len(rows):\n            temporary = manifest_file.with_suffix(".csv.tmp")\n            pd.DataFrame(recorded.values()).to_csv(temporary, index=False)\n            temporary.replace(manifest_file)\n            print(f"Pixel audit: {len(recorded)}/{len(rows)} images", flush=True)\n            if cache_images:\n                estimate = float(np.mean(cached_sizes)) * (len(rows) - len(recorded)) * 1.25\n                require(estimate + 2 * 1024**3 < shutil.disk_usage(output).free,\n                        "Estimated PNG cache exceeds free disk space. Use a larger volume, or a NEW output directory with cache_images=False.")\n    manifest = pd.DataFrame(recorded.values()).sort_values(["patient_id", "image_id"]).reset_index(drop=True)\n    require(len(manifest) == len(rows), "Incomplete image manifest.")\n    check_cross_patient_duplicates(manifest)\n    write_json(output / "pixel_audit_complete.json", {"metadata_sha256": source_hash, "n_images": len(manifest),\n                                                      "manifest_sha256": file_sha256(manifest_file),\n                                                      "cross_patient_exact_pixel_duplicates": 0})\n    return manifest\n')
target = PROJECT / 'gmic_pilot_analysis.py'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text('"""Explicit probability pooling for fixed external GMIC development models.\n\nThis is separate from the original logit-pooling ResNet pipeline. It does not\ncalibrate, choose a threshold, select a member, or estimate a final endpoint.\n"""\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import average_precision_score, roc_auc_score\nfrom mammo_reliability.audit import BREAST_KEY, require\n\nMEMBERS = list(range(1, 6))\nPREDICTION_COLUMNS = [f\'m{m}_{target}\' for m in MEMBERS for target in [\'benign\', \'malignant\']]\n\n\ndef aggregate_predictions(images, predictions):\n    """Average repeats within view, then give CC and MLO equal weight."""\n    require(images.image_id.is_unique and predictions.image_id.is_unique, \'Duplicate image identifiers.\')\n    require(set(images.image_id) == set(predictions.image_id), \'Incomplete or unexpected prediction coverage.\')\n    require(set(PREDICTION_COLUMNS).issubset(predictions), \'Missing model output columns.\')\n    values = predictions[PREDICTION_COLUMNS].to_numpy(dtype=float)\n    require(np.isfinite(values).all() and ((values >= 0) & (values <= 1)).all(), \'Invalid predictions.\')\n    require(images.view.isin([\'CC\', \'MLO\']).all(), \'Nonstandard view in the pilot.\')\n    require(images.role.isin([\'fit\', \'tune\']).all(), \'A nondevelopment role entered the pilot.\')\n    require(images.cancer.isin([0, 1]).all(), \'Invalid cancer labels.\')\n    groups = images.groupby(BREAST_KEY)\n    require((groups.view.nunique() == 2).all(), \'A breast lacks a standard view.\')\n    for column in [\'cancer\', \'site_id\', \'role\']:\n        require((groups[column].nunique(dropna=False) == 1).all(), f\'Conflicting {column}.\')\n    require((images.groupby(\'patient_id\').role.nunique() == 1).all(), \'Patient crosses pilot roles.\')\n    require(not set(PREDICTION_COLUMNS).intersection(images.columns), \'Predictions already in metadata.\')\n    joined = images.merge(predictions[[\'image_id\'] + PREDICTION_COLUMNS], on=\'image_id\', validate=\'one_to_one\')\n    malignant = [f\'m{m}_malignant\' for m in MEMBERS]\n    views = joined.groupby(BREAST_KEY + [\'view\'])[malignant].mean()\n    cc, mlo = views.xs(\'CC\', level=\'view\'), views.xs(\'MLO\', level=\'view\')\n    require(cc.index.equals(mlo.index), \'CC and MLO breast identities do not match.\')\n    result = groups.agg(site_id=(\'site_id\', \'first\'), role=(\'role\', \'first\'),\n                        cancer=(\'cancer\', \'first\'), n_images=(\'image_id\', \'size\'))\n    for member, column in zip(MEMBERS, malignant):\n        result[f\'m{member}_cc\'] = cc[column]\n        result[f\'m{member}_mlo\'] = mlo[column]\n        result[f\'m{member}_breast\'] = (cc[column] + mlo[column]) / 2\n    members = result[[f\'m{m}_breast\' for m in MEMBERS]].to_numpy()\n    result[\'p_raw\'] = members.mean(axis=1)\n    result[\'p_cc\'] = cc.mean(axis=1)\n    result[\'p_mlo\'] = mlo.mean(axis=1)\n    result[\'view_disagreement\'] = abs(result.p_cc - result.p_mlo)\n    result[\'ensemble_variance\'] = members.var(axis=1, ddof=0)\n    require(np.isfinite(result.p_raw).all(), \'Missing pooled predictions.\')\n    return result.reset_index()\n\n\ndef rank_metrics(frame, column):\n    y, p = frame.cancer.to_numpy(), frame[column].to_numpy()\n    return {\'breasts\': len(frame), \'positive_breasts\': int(y.sum()),\n            \'prevalence\': float(y.mean()),\n            \'average_precision\': float(average_precision_score(y, p)) if y.sum() else None,\n            \'roc_auc\': float(roc_auc_score(y, p)) if len(np.unique(y)) == 2 else None}\n\n\ndef summarize_development(breasts):\n    rows = []\n    for role, group in breasts.groupby(\'role\', sort=True):\n        # All fixed members are reported; no best-member selection is performed.\n        for name, column in [(\'ensemble\', \'p_raw\')] + [(f\'model_{m}\', f\'m{m}_breast\') for m in MEMBERS]:\n            rows.append({\'role\': role, \'model\': name, \'site\': \'pooled\', **rank_metrics(group, column)})\n        for site, subset in group.groupby(\'site_id\'):\n            rows.append({\'role\': role, \'model\': \'ensemble\', \'site\': int(site), **rank_metrics(subset, \'p_raw\')})\n    return {\'purpose\': \'adaptive_reserved_development_only\', \'model_fitted_on_rsna\': False,\n            \'threshold_selected\': False, \'calibration_fitted\': False,\n            \'final_test_results_opened\': False, \'metrics\': rows,\n            \'interpretation\': \'Released RSNA labels; development estimates after prior adaptive trials. Not final performance.\'}\n')
target = PROJECT / 'gmic_operating_point.py'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text('"""Development-only threshold and failure-review analysis of fixed GMIC scores.\n\nRun from the project directory. No image loading, GPU, network, model selection,\nor final-cohort inference occurs. See GMIC_OPERATING_POINT_PLAN.md.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport platform\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport scipy\nimport sklearn\nfrom sklearn.metrics import brier_score_loss, log_loss\n\nfrom gmic_pilot_analysis import aggregate_predictions, summarize_development\nfrom mammo_reliability.audit import BREAST_KEY, require\nfrom mammo_reliability.evaluation import (\n    MonotonePlatt, add_review_scores, classification_metrics,\n    paired_cluster_bootstrap, review_at_budget, review_selection,\n    sensitivity_threshold,\n)\n\nSOURCE_URL = \'https://www.kaggle.com/code/aseqwss/11-rsna-gmic-reserved-pilot/output?scriptVersionId=352364754\'\nEXPECTED_PLAN = \'388b2f20d08e84850e8ae14b6d8e3ad9848da9bb4a76a061a48ea11c27a913ba\'\nEXPECTED_BREASTS = \'eb2bf94e1e752adb9cd6f96ec06898905acaf40815958f52e14a5a72df7bb906\'\nEXPECTED_ROLES = \'fc0308c63300be6849c7ac9b830d09480c2b62d64df4f1d036bab12548a7e15f\'\nMETHODS = [\'score_confidence\', \'score_disagreement\', \'score_uncertainty\',\n           \'score_confidence_disagreement\', \'score_confidence_uncertainty\', \'score_combined\']\nBUDGETS = [0.01, 0.05, 0.10, 0.20]\n\n\ndef digest(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef load_verified(root: Path):\n    complete = json.loads((root / \'completed.json\').read_text())\n    require(complete[\'status\'] == \'reserved_development_inference_completed\', \'Incomplete inference.\')\n    require(complete[\'final_test_results_opened\'] is False, \'Final results entered the source.\')\n    require(complete[\'declared_plan_sha256\'] == EXPECTED_PLAN, \'Unexpected pilot plan.\')\n    require(complete[\'breast_predictions_sha256\'] == EXPECTED_BREASTS, \'Unexpected scored cohort.\')\n    for key, name in [(\'declared_plan_sha256\', \'declared_plan.json\'),\n                      (\'image_predictions_sha256\', \'image_predictions_private.csv\'),\n                      (\'breast_predictions_sha256\', \'breast_predictions_private.csv\'),\n                      (\'summary_sha256\', \'development_summary.json\')]:\n        require(digest(root / name) == complete[key], f\'File integrity failed: {name}\')\n    require(digest(root / \'roles_private.csv\') == EXPECTED_ROLES, \'Patient roles changed.\')\n    plan = json.loads((root / \'declared_plan.json\').read_text())\n    project = Path(__file__).resolve().parent\n    for name, expected in plan[\'source_sha256\'].items():\n        require(digest(project / name) == expected, f\'Inference source changed: {name}\')\n    roles = pd.read_csv(root / \'roles_private.csv\')\n    images = pd.read_csv(root / \'images_private.csv\')\n    predictions = pd.read_csv(root / \'image_predictions_private.csv\')\n    breasts = pd.read_csv(root / \'breast_predictions_private.csv\')\n    require(roles.patient_id.is_unique and len(roles) == 2383, \'Invalid role table.\')\n    require(set(roles.cohort) == {\'pilot\'}, \'Nonpilot patient in source.\')\n    require(set(images.patient_id) == set(roles.patient_id) == set(breasts.patient_id), \'Patient coverage differs.\')\n    expected_roles = roles.set_index(\'patient_id\').role\n    for frame in [images, breasts]:\n        require((frame.role == frame.patient_id.map(expected_roles)).all(), \'Patient-role mismatch.\')\n    require(len(images) == 10881 and len(breasts) == 4766, \'Unexpected cohort size.\')\n    recomputed = aggregate_predictions(images, predictions)\n    keys = BREAST_KEY\n    ordered = breasts.sort_values(keys).reset_index(drop=True)\n    recomputed = recomputed[ordered.columns].sort_values(keys).reset_index(drop=True)\n    pd.testing.assert_frame_equal(ordered, recomputed, check_exact=False, atol=1e-12, rtol=1e-12)\n    for role, size, positives in [(\'fit\', 3574, 73), (\'tune\', 1192, 26)]:\n        group = breasts.loc[breasts.role == role]\n        require(len(group) == size and int(group.cancer.sum()) == positives, \'Unexpected role counts.\')\n    require(not set(breasts.loc[breasts.role == \'fit\', \'patient_id\']) &\n            set(breasts.loc[breasts.role == \'tune\', \'patient_id\']), \'Patient leakage.\')\n    original = json.loads((root / \'development_summary.json\').read_text())\n    current = summarize_development(breasts)\n    for a, b in zip(original[\'metrics\'], current[\'metrics\'], strict=True):\n        for name in [\'role\', \'model\', \'site\', \'breasts\', \'positive_breasts\']:\n            require(a[name] == b[name], \'Original metric group mismatch.\')\n        for name in [\'average_precision\', \'roc_auc\']:\n            require(abs(a[name] - b[name]) < 1e-12, \'Original ranking metric failed to reproduce.\')\n    return breasts, {\'source\': SOURCE_URL, \'complete\': complete,\n                     \'input_sha256\': {p.name: digest(p) for p in sorted(root.iterdir()) if p.is_file()},\n                     \'pooling_and_ranking_reproduced\': True, \'geometry\': original[\'geometry\']}\n\n\ndef fit_development_rule(fit):\n    require(len(fit) > 0 and set(fit.role) == {\'fit\'}, \'Fit-only calibration is required.\')\n    calibrator = MonotonePlatt().fit(fit.p_raw, fit.cancer)\n    reference = fit.copy()\n    reference[\'p_calibrated\'] = calibrator.predict(reference.p_raw)\n    threshold = sensitivity_threshold(reference.p_calibrated, reference.cancer, target=0.90)\n    raw_threshold = sensitivity_threshold(reference.p_raw, reference.cancer, target=0.90)\n    require(np.array_equal(reference.p_raw >= raw_threshold, reference.p_calibrated >= threshold),\n            \'Calibration changed operating-point decisions.\')\n    return calibrator, reference, threshold, raw_threshold\n\n\ndef apply_development_rule(calibrator, reference, target, threshold):\n    require(not set(reference.patient_id) & set(target.patient_id), \'Fit/tune patient overlap.\')\n    require(set(target.role) == {\'tune\'}, \'Evaluation is restricted to tune patients.\')\n    scored = target.copy()\n    scored[\'p_calibrated\'] = calibrator.predict(scored.p_raw)\n    return add_review_scores(reference, scored, threshold)\n\n\ndef subgroup_reviews(scored, method, budget):\n    """Subgroups inherit the one global selection; no subgroup budget refitting."""\n    selected = review_selection(scored, method, budget)\n    rows = []\n    for site, group in scored.groupby(\'site_id\'):\n        chosen = selected.loc[group.index]\n        negatives = group.loc[~group.predicted_positive]\n        fn, caught = int(negatives.cancer.sum()), int(group.loc[chosen, \'cancer\'].sum())\n        k = int(chosen.sum())\n        rows.append({\'site_id\': int(site), \'method\': method, \'global_budget\': budget,\n                     \'negative_breasts\': len(negatives), \'false_negatives\': fn,\n                     \'reviewed_breasts\': k, \'false_negatives_flagged\': caught,\n                     \'fn_capture_fraction\': caught / fn if fn else None,\n                     \'within_site_review_fraction\': k / len(negatives) if len(negatives) else None,\n                     \'selection\': \'inherited from global cohort; no site-specific reallocation\'})\n    return rows\n\n\ndef analyze(breasts, repeats=2000):\n    fit = breasts.loc[breasts.role == \'fit\'].copy().reset_index(drop=True)\n    tune = breasts.loc[breasts.role == \'tune\'].copy().reset_index(drop=True)\n    calibrator, reference, threshold, raw_threshold = fit_development_rule(fit)\n    scored = apply_development_rule(calibrator, reference, tune, threshold)\n    require(np.array_equal(scored.predicted_positive, scored.p_raw >= raw_threshold), \'Ranking/threshold mismatch.\')\n    metrics, calibration, distributions = [], [], []\n    prior = float(fit.cancer.mean())\n    for role, frame in [(\'fit\', reference), (\'tune\', scored)]:\n        groups = [(\'pooled\', frame)] + [(int(site), group) for site, group in frame.groupby(\'site_id\')]\n        for site, group in groups:\n            metrics.append({\'role\': role, \'site\': site, **classification_metrics(group.cancer, group.p_calibrated, threshold)})\n            for name, values in [(\'raw\', group.p_raw), (\'calibrated\', group.p_calibrated),\n                                 (\'constant_fit_prevalence\', np.full(len(group), prior))]:\n                calibration.append({\'role\': role, \'site\': site, \'score\': name,\n                    \'brier\': float(brier_score_loss(group.cancer, values)),\n                    \'log_loss\': float(log_loss(group.cancer, values, labels=[0, 1]))})\n            for column in [\'p_raw\', \'view_disagreement\', \'ensemble_variance\']:\n                distributions.append({\'role\': role, \'site\': site, \'column\': column,\n                    \'quantiles\': {str(q): float(group[column].quantile(q)) for q in [0, .1, .5, .9, .99, 1]}})\n    review, site_review = [], []\n    for budget in BUDGETS:\n        for method in METHODS:\n            row = review_at_budget(scored, method, budget)\n            row[\'review_fraction_all_tune_breasts\'] = row[\'reviewed_breasts\'] / len(scored)\n            review.append(row)\n            site_review.extend(subgroup_reviews(scored, method, budget))\n    # One development evaluation group, not an outer test fold. This constant\n    # reuses the fold-budget bootstrap without splitting or double-counting it.\n    bootstrap_frame = scored.assign(outer_fold=0)\n    bootstrap = paired_cluster_bootstrap(bootstrap_frame, repeats=repeats)\n    bootstrap[\'estimand\'] = \'development tune-cohort combined-minus-cancer-score FN capture at a global 5% review budget\'\n    negative = scored.loc[~scored.predicted_positive]\n    correlation = negative[[\'p_raw\', \'view_disagreement\', \'ensemble_variance\']].corr(method=\'spearman\')\n    result = {\'purpose\': \'adaptive_development_operating_point_feasibility\',\n        \'final_test_results_opened\': False, \'target_sensitivity\': .90,\n        \'calibration_fit_role\': \'fit\', \'evaluation_role\': \'tune\',\n        \'raw_threshold\': raw_threshold, \'calibrated_threshold\': threshold,\n        \'calibrator\': {\'slope\': calibrator.slope, \'intercept\': calibrator.intercept},\n        \'normalization_reference_breasts\': int((reference.p_calibrated < threshold).sum()),\n        \'classification\': metrics, \'calibration_comparison\': calibration,\n        \'review\': review, \'site_review\': site_review, \'primary_development_bootstrap\': bootstrap,\n        \'score_distributions\': distributions,\n        \'negative_tune_spearman\': correlation.to_dict(),\n        \'limitations\': [\'Adaptive development, not final performance or a novelty claim.\',\n            \'Only 26 tune-positive breasts; subgroup and FN estimates are especially uncertain.\',\n            \'Threshold uncertainty and model selection are not included in the conditional bootstrap.\',\n            \'Flagged cases are simulated review candidates, not clinically recovered cancers.\',\n            \'Density and hormone-receptor policies are not evaluated here.\']}\n    return result, scored\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--input\', type=Path, required=True)\n    parser.add_argument(\'--output\', type=Path, default=Path(\'validation/gmic_operating_point\'))\n    parser.add_argument(\'--private-output\', type=Path, default=Path(\'private_runs/gmic_operating_point\'))\n    args = parser.parse_args()\n    breasts, provenance = load_verified(args.input)\n    result, scored = analyze(breasts)\n    result[\'provenance\'] = provenance\n    result[\'software\'] = {\'python\': platform.python_version(), \'numpy\': np.__version__,\n                          \'pandas\': pd.__version__, \'scipy\': scipy.__version__, \'sklearn\': sklearn.__version__}\n    result[\'analysis_sha256\'] = digest(__file__)\n    result[\'analysis_plan_sha256\'] = digest(Path(__file__).with_name(\'GMIC_OPERATING_POINT_PLAN.md\'))\n    args.output.mkdir(parents=True, exist_ok=True)\n    args.private_output.mkdir(parents=True, exist_ok=True)\n    (args.output / \'results.json\').write_text(json.dumps(result, indent=2, allow_nan=False) + \'\\n\')\n    for name in [\'classification\', \'calibration_comparison\', \'review\', \'site_review\']:\n        pd.DataFrame(result[name]).to_csv(args.output / f\'{name}.csv\', index=False)\n    scored.to_csv(args.private_output / \'scored_tune_private.csv\', index=False)\n    for row in result[\'classification\']:\n        if row[\'role\'] == \'tune\': print(\'TUNE\', json.dumps(row))\n    for row in result[\'review\']:\n        if row[\'budget\'] == .05: print(\'REVIEW_5_PERCENT\', json.dumps(row))\n    print(\'PAIRED_BOOTSTRAP\', json.dumps(result[\'primary_development_bootstrap\']))\n\n\nif __name__ == \'__main__\':\n    main()\n')
target = PROJECT / 'GMIC_OPERATING_POINT_PLAN.md'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text('# Reserved-development operating-point analysis — 2026-09-28\n\nDeclared after reviewing notebook 11 ensemble/member AP and ROC-AUC, before calculating its operating thresholds or false-negative review results. This is adaptive development, not preregistration or final validation.\n\n## Scope and fixed choices\n\n- Use only notebook 11 version 2 (352364754), completed successfully on the 2,383 reserved development patients. Verify saved file hashes and exact fit/tune counts. No final-evaluation patient may enter this analysis.\n- Retain all five external GMIC members, their original equal probability pooling, all scored breasts and both sites. Do not select a member, site, view or patient based on outcomes.\n- Fit one pooled positive-slope Platt calibrator using the 3,574 fit breasts only. Estimate a global threshold on those same fit breasts at the previously proposed 90% sensitivity target, with a positive decision defined by score >= threshold. The target is a research choice, not a clinical recommendation or guarantee.\n- Apply the fitted calibrator and threshold unchanged to all 1,192 tune breasts. Report confusion counts, sensitivity, specificity, the number of AI-negative breasts, AP/AUC, Brier score and log loss. Report raw and calibrated Brier/log loss and compare with a constant probability equal to fit prevalence. A monotone calibrator cannot improve ranking.\n- Normalize review features using the empirical distributions among AI-negative fit breasts, irrespective of their cancer labels. Reuse the previously specified six methods: cancer score alone; absolute CC–MLO disagreement; ensemble probability variance; equal percentile averages of score + disagreement, score + variance, and all three.\n- Apply review budgets of 1%, 5%, 10% and 20% of AI-negative tune breasts, rounded down. Five percent remains the focal comparison. Use existing deterministic identifier-based tie ordering, never outcomes, to fill an exact budget.\n- Report the number of false negatives flagged, total false negatives, review yield and global review workload. A flag is a simulated review candidate; no radiologist review or clinical cancer recovery is measured.\n- Report site subgroups using the same global threshold and the same globally selected candidates. Do not separately allocate budgets or tune site-specific rules. Density remains a later exploratory analysis; this analysis will not infer missing density.\n- Calculate a paired patient-cluster bootstrap with 2,000 replicates for the combined-minus-score FN capture difference at 5%, retaining both breasts together. Fitted calibration and threshold stay fixed. Report undefined replicates and acknowledge that very few misses make the interval unstable. This interval is not a confirmatory hypothesis test after adaptive development.\n- Describe score distributions and correlations by role/site to detect practical concerns. Do not attribute site differences to breast density, device, or biology without supporting evidence.\n\n## Decision after analysis\n\nUse these results to assess whether a failure-prioritization study is feasible and whether the proposed uncertainty signals show any development promise beyond cancer score alone. Preserve negative results. Do not alter the final protocol or run final inference automatically. Adopting the fixed external ensemble requires an explicit prospective protocol revision and a substantive literature-gap review.\n')
target = PROJECT / 'gmic_orientation_review.py'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text('"""Generate private technical displays for every predeclared orientation flag."""\nfrom pathlib import Path\nimport json\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image, ImageDraw, ImageOps\nfrom gmic_adapter import decode_native, digest_file\nfrom gmic_fullfield import fullfield_array\nfrom mammo_reliability.audit import locate_dataset, require\n\n\ndef run(source, output):\n    source, output = Path(source), Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    plan = json.loads((source/\'declared_plan.json\').read_text())\n    predictions = pd.read_csv(source/\'image_predictions_private.csv\')\n    images = pd.read_csv(source/\'images_private.csv\')\n    selected = predictions.loc[predictions.edge_mass_contrast < plan[\'orientation_low_contrast_flag\']]\n    selected = selected.merge(images, on=\'image_id\', validate=\'one_to_one\').sort_values(\'image_id\')\n    require(len(selected) == 3 and set(selected.role) == {\'fit\'}, \'Unexpected orientation-review scope.\')\n    # Selection uses the previously recorded image statistic, never labels.\n    data_root = locate_dataset()\n    sheet = Image.new(\'RGB\', (1200, 2280), \'#171b22\')\n    draw = ImageDraw.Draw(sheet)\n    records = []\n    for index, row in enumerate(selected.itertuples(index=False), start=1):\n        path = data_root/\'train_images\'/str(row.patient_id)/f\'{row.image_id}.dcm\'\n        require(digest_file(path) == row.native_file_sha256, \'Native image bytes changed.\')\n        raw, header = decode_native(path)\n        _, canvas, geometry = fullfield_array(raw, f\'{row.laterality}-{row.view}\')\n        require(abs(geometry[\'edge_mass_contrast\'] - row.edge_mass_contrast) < 1e-10,\n                \'Orientation statistic did not reproduce.\')\n        require(geometry[\'actual_flip_applied\'] == row.actual_flip_applied, \'Orientation decision changed.\')\n        for col, (label, array) in enumerate([(\'Original full image\', raw), (\'Current full-field model input\', canvas)]):\n            # Display only: map the same 16-bit intensity scale to 8-bit and\n            # preserve aspect ratio. No independent contrast enhancement/crop.\n            display = Image.fromarray(np.clip(np.asarray(array)/65535*255, 0, 255).astype(np.uint8))\n            display = ImageOps.contain(display, (570, 700))\n            x, y = col*600+15, (index-1)*760\n            draw.text((x, y+8), f\'Flag {index} | {row.view} | {label}\', fill=\'white\')\n            draw.text((x, y+25), f\'Edge contrast {row.edge_mass_contrast:.4f}\', fill=\'white\')\n            sheet.paste(display, (x+(570-display.width)//2, y+48))\n        records.append({\'flag_index\': index, \'site_id\': int(row.site_id), \'role\': row.role,\n                        \'view\': row.view, \'geometry\': geometry, \'dicom\': header})\n    sheet.save(output/\'orientation_flags_private.png\')\n    (output/\'orientation_numeric_audit.json\').write_text(json.dumps({\n        \'selection\': \'All images below predeclared edge-mass contrast 0.2; labels not used\',\n        \'images\': len(records), \'visual_review_completed\': False, \'records\': records,\n        \'no_predictions_changed\': True, \'final_test_results_opened\': False}, indent=2)+\'\\n\')\n    print(\'ORIENTATION_REVIEW_PREPARED\', json.dumps({\'images\': len(records), \'role\': \'fit\',\n          \'no_predictions_changed\': True, \'visual_review_completed\': False}))\n')
target = PROJECT / 'gmic_pilot.py'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text('"""Fixed external mammography ensemble; reserved development patients only."""\nfrom pathlib import Path\nimport csv\nimport json\nimport os\nimport shutil\nimport time\nimport numpy as np\nimport pandas as pd\nimport torch\n\nfrom gmic_adapter import decode_native, digest_file\nfrom gmic_fullfield import fullfield_array\nfrom gmic_preflight import infer\nfrom gmic_fullfield_preflight import verified_preflight, load_fixed_models\nfrom gmic_pilot_analysis import aggregate_predictions, summarize_development, PREDICTION_COLUMNS\nfrom mammo_reliability.audit import locate_dataset, require, write_json, environment\nfrom mammo_reliability.training import source_fingerprint\n\nOUTPUT = Path(\'/kaggle/working/gmic_reserved_pilot\')\nEXPECTED_ROLES = \'fc0308c63300be6849c7ac9b830d09480c2b62d64df4f1d036bab12548a7e15f\'\nEXPECTED_METADATA = \'35408731b67b87272f85874ee3c4c9d5cea7a2c4fbcaeac3fbe415ba1a5817c1\'\n\n\ndef verified_geometry():\n    candidates = [Path(\'/kaggle/input/notebooks/aseqwss/10-rsna-gmic-fullfield-check/gmic_fullfield_preflight\'),\n                  Path(\'/kaggle/input/10-rsna-gmic-fullfield-check/gmic_fullfield_preflight\')]\n    existing = [p for p in candidates if p.is_dir()]\n    require(len(existing) == 1, \'Attach notebook 10 with the reviewed geometry audit.\')\n    root = existing[0]\n    review = json.loads(Path(\'gmic_visual_review.json\').read_text())\n    require(review[\'engineering_visual_review_passed\'] is True and review[\'images_reviewed\'] == 20,\n            \'Complete technical visual review first.\')\n    require(digest_file(root/\'declared_plan.json\') == review[\'declared_plan_sha256\'], \'Unreviewed audit version.\')\n    complete = json.loads((root/\'completed.json\').read_text())\n    require(complete[\'images\'] == 20 and not complete[\'final_test_results_opened\'], \'Incomplete image audit.\')\n    declaration = json.loads((root/\'declared_plan.json\').read_text())\n    for key, name in [(\'fullfield_sha256\', \'gmic_fullfield.py\'), (\'adapter_sha256\', \'gmic_adapter.py\')]:\n        require(declaration[key] == digest_file(name), f\'Unreviewed processing changes: {name}\')\n    return root, review\n\n\ndef main():\n    started = time.monotonic()\n    OUTPUT.mkdir(parents=True, exist_ok=True)\n    # A new run must never accidentally combine partial outputs with new code.\n    require(not (OUTPUT/\'image_predictions_private.csv\').exists(), \'Existing predictions require explicit verified recovery.\')\n    require(torch.cuda.is_available(), \'GPU required.\')\n    require(source_fingerprint() == \'f4178761b59233ffe1048218d234624c22764c8129d11c48fe30821cbe079a7c\',\n            \'Original model package changed.\')\n    torch.manual_seed(17)\n    np.random.seed(17)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.allow_tf32 = False\n    base, upstream = verified_preflight()\n    geometry_root, review = verified_geometry()\n    roles_path = base/\'cohort_identity/pilot/roles_private.csv\'\n    require(digest_file(roles_path) == EXPECTED_ROLES, \'Patient roles changed.\')\n    roles = pd.read_csv(roles_path)\n    require(roles.patient_id.is_unique and set(roles.cohort) == {\'pilot\'} and (roles.outer_fold == -1).all(),\n            \'A final evaluation patient entered development.\')\n    require(set(roles.role) == {\'fit\', \'tune\'} and len(roles) == 2383, \'Wrong development roles.\')\n    data_root = locate_dataset()\n    require(digest_file(data_root/\'train.csv\') == EXPECTED_METADATA, \'Metadata changed.\')\n    all_images = pd.read_csv(data_root/\'train.csv\')\n    images = all_images.loc[all_images.patient_id.isin(roles.patient_id) & all_images.view.isin([\'CC\', \'MLO\'])]\n    images = images.merge(roles[[\'patient_id\', \'role\']], on=\'patient_id\', validate=\'many_to_one\')\n    images = images.sort_values([\'patient_id\', \'laterality\', \'view\', \'image_id\']).reset_index(drop=True)\n    require(len(images) == 10881 and images.image_id.is_unique, \'Wrong pilot image coverage.\')\n    require(set(images.patient_id) == set(roles.patient_id), \'Missing development patients.\')\n    require(len(images[[\'patient_id\', \'laterality\']].drop_duplicates()) == 4766, \'Wrong breast count.\')\n    # The labels below are used only in the final descriptive analysis.\n    images.to_csv(OUTPUT/\'images_private.csv\', index=False)\n    shutil.copyfile(roles_path, OUTPUT/\'roles_private.csv\')\n    shutil.copyfile(\'gmic_visual_review.json\', OUTPUT/\'visual_review.json\')\n    declaration = {\n        \'purpose\': \'reserved_development_transfer_feasibility\', \'rsna_model_training\': False,\n        \'patients\': 2383, \'breasts\': 4766, \'images\': 10881,\n        \'roles_sha256\': EXPECTED_ROLES, \'metadata_sha256\': EXPECTED_METADATA,\n        \'upstream_commit\': \'594029efe6184bcca3a151fa1dd46c5fd620cb25\',\n        \'models\': [1, 2, 3, 4, 5], \'class_order\': [\'benign\', \'malignant\'],\n        \'inference\': \'FP32, one deterministic full-field image, fixed external weights, no TTA\',\n        \'pooling\': \'arithmetic malignant probability mean within repeated view; equal CC/MLO mean; equal five-model mean\',\n        \'uncertainty_descriptors\': \'absolute ensemble CC-MLO difference; population variance of five breast probabilities\',\n        \'analysis\': \'AP and ROC-AUC for all fixed members and ensemble by pilot role; ensemble also by site\',\n        \'threshold_selection\': False, \'calibration\': False, \'density_effect_testing\': False,\n        \'final_test_results_opened\': False, \'model_or_member_selection_in_job\': False,\n        \'orientation_low_contrast_flag\': .2, \'orientation_flags_exclude_images\': False,\n        \'environment\': environment(), \'gpu\': torch.cuda.get_device_name(0),\n        \'source_sha256\': {p: digest_file(p) for p in [\n            \'gmic_pilot.py\', \'gmic_pilot_analysis.py\', \'gmic_adapter.py\', \'gmic_preflight.py\',\n            \'gmic_fullfield.py\', \'gmic_fullfield_preflight.py\', \'gmic_visual_review.json\']},\n        \'geometry_review\': review,\n        \'limitations\': [\'Adapted image scale can reduce small-feature visibility.\',\n                       \'Heuristic orientation is not validated for every image.\',\n                       \'Five published models were selected in external hyperparameter searches, not a seed-only ensemble.\',\n                       \'Different reported source institutions do not prove patient-level independence.\',\n                       \'Adaptive development estimates are not final research results.\']}\n    write_json(OUTPUT/\'declared_plan.json\', declaration)\n    print(\'GMIC PILOT DECLARED PLAN\', json.dumps(declaration), flush=True)\n    models = load_fixed_models(upstream, OUTPUT)\n    # Reproduce every model on the image that exposed the earlier crop failure.\n    # A pandas Series has a .view method; use a named row for the view column.\n    sample = next(pd.read_csv(base/\'engineering_sample_private.csv\').iloc[[10]].itertuples(index=False))\n    require(sample.patient_id in set(roles.loc[roles.role.eq(\'fit\'), \'patient_id\']), \'Probe is not a fitting image.\')\n    raw, _ = decode_native(data_root/\'train_images\'/str(sample.patient_id)/f\'{sample.image_id}.dcm\')\n    array, _, _ = fullfield_array(raw, f\'{sample.laterality}-{sample.view}\')\n    probe = infer(models, array)\n    expected = np.load(geometry_root/\'sample_11_predictions_private.npy\', allow_pickle=False)\n    error = float(np.max(abs(probe-expected)))\n    require(error <= 1e-6, \'The reviewed image predictions did not reproduce.\')\n    write_json(OUTPUT/\'reproduction_probe.json\', {\'sample\': 11, \'max_absolute_error\': error, \'tolerance\': 1e-6})\n    print(\'GMIC PILOT REPRODUCTION\', json.dumps({\'sample\': 11, \'max_absolute_error\': error}), flush=True)\n    fields = [\'image_id\'] + PREDICTION_COLUMNS + [\'scale\', \'edge_mass_contrast\', \'actual_flip_applied\',\n                                                \'raw_height\', \'raw_width\', \'native_file_sha256\']\n    with (OUTPUT/\'image_predictions_private.csv\').open(\'w\', newline=\'\') as stream:\n        writer = csv.DictWriter(stream, fieldnames=fields)\n        writer.writeheader()\n        for count, row in enumerate(images.itertuples(index=False), 1):\n            path = data_root/\'train_images\'/str(row.patient_id)/f\'{row.image_id}.dcm\'\n            raw, _ = decode_native(path)\n            array, _, geometry = fullfield_array(raw, f\'{row.laterality}-{row.view}\')\n            predictions = infer(models, array)\n            record = {\'image_id\': row.image_id, **dict(zip(PREDICTION_COLUMNS, predictions.ravel().tolist())),\n                      \'scale\': geometry[\'scale\'], \'edge_mass_contrast\': geometry[\'edge_mass_contrast\'],\n                      \'actual_flip_applied\': geometry[\'actual_flip_applied\'],\n                      \'raw_height\': raw.shape[0], \'raw_width\': raw.shape[1], \'native_file_sha256\': digest_file(path)}\n            writer.writerow(record)\n            if count % 100 == 0 or count == len(images):\n                stream.flush(); os.fsync(stream.fileno())\n                progress = {\'images_completed\': count, \'images_total\': len(images),\n                            \'elapsed_seconds\': time.monotonic()-started, \'final_test_results_opened\': False}\n                write_json(OUTPUT/\'progress.json\', progress)\n                print(\'GMIC PILOT PROGRESS\', json.dumps(progress), flush=True)\n    predictions = pd.read_csv(OUTPUT/\'image_predictions_private.csv\')\n    breasts = aggregate_predictions(images, predictions)\n    require(len(breasts) == 4766 and int(breasts.cancer.sum()) == 99, \'Wrong outcome coverage.\')\n    require(breasts.groupby(\'role\').size().to_dict() == {\'fit\': 3574, \'tune\': 1192}, \'Wrong role coverage.\')\n    breasts.to_csv(OUTPUT/\'breast_predictions_private.csv\', index=False)\n    summary = summarize_development(breasts)\n    summary[\'geometry\'] = {\'minimum_scale\': float(predictions.scale.min()),\n                           \'maximum_scale\': float(predictions.scale.max()),\n                           \'low_orientation_contrast_images\': int((predictions.edge_mass_contrast < .2).sum()),\n                           \'minimum_edge_mass_contrast\': float(predictions.edge_mass_contrast.min())}\n    write_json(OUTPUT/\'development_summary.json\', summary)\n    for record in summary[\'metrics\']: print(\'GMIC PILOT METRICS\', json.dumps(record), flush=True)\n    result = {\'status\': \'reserved_development_inference_completed\', \'images\': len(images), \'breasts\': len(breasts),\n              \'elapsed_seconds\': time.monotonic()-started, \'final_test_results_opened\': False,\n              \'declared_plan_sha256\': digest_file(OUTPUT/\'declared_plan.json\'),\n              \'image_predictions_sha256\': digest_file(OUTPUT/\'image_predictions_private.csv\'),\n              \'breast_predictions_sha256\': digest_file(OUTPUT/\'breast_predictions_private.csv\'),\n              \'summary_sha256\': digest_file(OUTPUT/\'development_summary.json\')}\n    write_json(OUTPUT/\'completed.json\', result)\n    print(\'GMIC PILOT COMPLETED\', json.dumps(result), flush=True)\n\n\nif __name__ == \'__main__\': main()\n')
target = PROJECT / 'gmic_adapter.py'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text('"""Engineering adapter for a pinned, unmodified NYU GMIC implementation.\n\nThe upstream implementation and weights retain their authorship and AGPLv3\nlicense. This adapter is development code; it is not a clinical pipeline.\n"""\nfrom pathlib import Path\nimport hashlib\nimport json\nimport shutil\nimport urllib.request\n\nimport numpy as np\n\nUPSTREAM_COMMIT = \'594029efe6184bcca3a151fa1dd46c5fd620cb25\'\nHISTORICAL_FIXTURE_COMMIT = \'febe2431dbe6efd75bcb37ddea85dc48c0df0b07\'\n\n\ndef filter_legacy_state(state, model_keys):\n    """Permit one unused saved tensor, never arbitrary missing/extra weights.\n\n    The pinned source has no shared_rep_filter module or forward-path use.\n    Its official loader uses strict=False. Our narrower exception preserves\n    strict loading for all live model parameters.\n    """\n    import copy\n    extra = set(state) - set(model_keys)\n    if extra - {\'shared_rep_filter.weight\'}:\n        raise ValueError(f\'Unexplained checkpoint tensors: {sorted(extra)}\')\n    missing = set(model_keys) - set(state)\n    if any(not name.endswith(\'.num_batches_tracked\') for name in missing):\n        raise ValueError(f\'Missing active checkpoint tensors: {sorted(missing)}\')\n    filtered = copy.copy(state)\n    for name in extra:\n        del filtered[name]\n    return filtered, sorted(extra)\n\n\ndef digest_file(path, algorithm=\'sha256\', git_blob=False):\n    path = Path(path)\n    digest = hashlib.new(algorithm)\n    if git_blob:\n        digest.update(b\'blob \' + str(path.stat().st_size).encode() + b\'\\0\')\n    with path.open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(1024*1024), b\'\'):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef acquire_upstream(manifest_path, destination):\n    """Check every file against the blob identity in the inspected Git tree."""\n    manifest = json.loads(Path(manifest_path).read_text())\n    if manifest[\'commit\'] not in {UPSTREAM_COMMIT, HISTORICAL_FIXTURE_COMMIT}:\n        raise ValueError(\'Unexpected GMIC revision.\')\n    destination = Path(destination)\n    records = []\n    for relative, blob in manifest[\'files\'].items():\n        relative_path = Path(relative)\n        if relative_path.is_absolute() or \'..\' in relative_path.parts:\n            raise ValueError(\'Invalid upstream path.\')\n        path = destination / relative_path\n        path.parent.mkdir(parents=True, exist_ok=True)\n        if not path.exists():\n            url = f"https://raw.githubusercontent.com/nyukat/GMIC/{manifest[\'commit\']}/{relative}"\n            temporary = path.with_suffix(path.suffix + \'.download\')\n            with urllib.request.urlopen(url, timeout=120) as source, temporary.open(\'wb\') as target:\n                shutil.copyfileobj(source, target, length=1024*1024)\n            if digest_file(temporary, \'sha1\', git_blob=True) != blob:\n                raise ValueError(f\'Upstream content mismatch: {relative}\')\n            temporary.replace(path)\n        if digest_file(path, \'sha1\', git_blob=True) != blob:\n            raise ValueError(f\'Existing upstream content mismatch: {relative}\')\n        records.append({\'path\': relative, \'git_blob\': blob, \'bytes\': path.stat().st_size,\n                        \'sha256\': digest_file(path)})\n        print(\'GMIC VERIFIED FILE\', relative, path.stat().st_size, flush=True)\n    return records\n\n\ndef infer_horizontal_flip(image, view):\n    """Provisional, label-free orientation heuristic requiring visual review.\n\nCompare intensity mass in outer fifths of the original image. The brighter\nside is the proposed chest-wall side. This is NOT a validated orientation\ndetector. Ambiguity is recorded, not used to exclude patients silently.\n"""\n    image = np.asarray(image)\n    if image.ndim != 2 or min(image.shape) < 5 or view not in (\'L-CC\', \'R-CC\', \'L-MLO\', \'R-MLO\'):\n        raise ValueError(\'Invalid image or standard view.\')\n    if not np.isfinite(image).all() or image.min() < 0 or image.max() <= image.min():\n        raise ValueError(\'Invalid image intensity range.\')\n    width = max(1, image.shape[1] // 5)\n    left = float(image[:, :width].sum(dtype=np.float64))\n    right = float(image[:, -width:].sum(dtype=np.float64))\n    total = left + right\n    if total <= 0:\n        raise ValueError(\'No edge signal for orientation.\')\n    wall_left = left >= right\n    usual_wall_left = view.startswith(\'L-\')\n    return (\'NO\' if wall_left == usual_wall_left else \'YES\'), {\n        \'proposed_chest_wall_side\': \'left\' if wall_left else \'right\',\n        \'edge_mass_contrast\': abs(left-right)/total,\n        \'left_edge_mass\': left, \'right_edge_mass\': right,\n        \'orientation_requires_visual_review\': True,\n    }\n\n\ndef preprocess_array(image, view, horizontal_flip):\n    """Use original crop, center, orientation, padding and normalization code."""\n    from src.cropping.crop_mammogram import crop_img_from_largest_connected, image_orientation\n    from src.optimal_centers.get_optimal_centers import extract_center\n    from src.data_loading import loading, augmentations\n\n    info = crop_img_from_largest_connected(\n        image, image_orientation(horizontal_flip, view[0]), True, 100, 50, 1/3)\n    top, bottom, left, right = map(int, info[0])\n    crop = image[top:bottom, left:right]\n    metadata = {\'full_view\': view, \'view\': view[2:], \'side\': view[0],\n                \'horizontal_flip\': horizontal_flip, \'window_location\': info[0],\n                \'rightmost_points\': info[1], \'bottommost_points\': info[2],\n                \'distance_from_starting_side\': info[3]}\n    center = extract_center(metadata, crop)\n    oriented = loading.flip_image(crop.astype(np.float32), view, horizontal_flip)\n    window, _ = augmentations.random_augmentation_best_center(\n        oriented, (2944, 1920), np.random.RandomState(0), best_center=center, view=view)\n    # Independently check the helper against the exact upstream inference entry.\n    processed = loading.process_image(oriented, view, center)\n    expected = window.copy()\n    loading.standard_normalize_single_image(expected)\n    if not np.array_equal(processed, expected):\n        raise ValueError(\'Adapter changed upstream image processing.\')\n    if processed.shape != (2944, 1920) or not np.isfinite(processed).all() or processed.std() < .99:\n        raise ValueError(\'Invalid processed GMIC input.\')\n    nonzero = max(int(np.count_nonzero(image)), 1)\n    details = {\'raw_shape\': list(image.shape), \'crop_shape\': list(crop.shape),\n               \'window_location\': [top, bottom, left, right],\n               \'best_center\': [int(x) for x in center],\n               \'distance_from_starting_side\': int(info[3]),\n               \'nonzero_fraction_retained_after_crop\': float(np.count_nonzero(crop)/nonzero),\n               \'nonzero_fraction_retained_in_window\': float(np.count_nonzero(window)/nonzero),\n               \'horizontal_flip\': horizontal_flip,\n               \'normalized_mean\': float(processed.mean()), \'normalized_std\': float(processed.std()),\n               \'processed_sha256\': hashlib.sha256(processed.tobytes()).hexdigest()}\n    return processed, window, details\n\n\ndef decode_native(path):\n    """Keep native pixel dimensions; no reuse/upscaling of the small cache."""\n    import pydicom\n    from pydicom.pixels import apply_modality_lut, apply_voi_lut\n    from mammo_reliability.imaging import normalize_pixels\n    ds = pydicom.dcmread(path)\n    raw = ds.pixel_array\n    if raw.ndim == 3 and raw.shape[0] == 1:\n        raw = raw[0]\n    if raw.ndim != 2 or \'PresentationLUTSequence\' in ds:\n        raise ValueError(\'Unsupported DICOM presentation.\')\n    normalized = normalize_pixels(raw, apply_voi_lut(apply_modality_lut(raw, ds), ds),\n                                  str(ds.PhotometricInterpretation), ds.get(\'PixelPaddingValue\'),\n                                  ds.get(\'PixelPaddingRangeLimit\'))\n    image = np.round(normalized*65535).astype(np.uint16)\n    details = {\'photometric\': str(ds.PhotometricInterpretation),\n               \'transfer_syntax\': str(ds.file_meta.TransferSyntaxUID),\n               \'patient_orientation\': list(ds.get(\'PatientOrientation\', [])),\n               \'pixel_spacing\': [float(x) for x in ds.get(\'PixelSpacing\', [])],\n               \'imager_pixel_spacing\': [float(x) for x in ds.get(\'ImagerPixelSpacing\', [])],\n               \'manufacturer\': str(ds.get(\'Manufacturer\', \'\')),\n               \'voi_present\': bool(\'VOILUTSequence\' in ds or \'WindowCenter\' in ds)}\n    return image, details\n')
target = PROJECT / 'gmic_preflight.py'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text('"""Reference reproduction and fit-only engineering check, no RSNA accuracy claim."""\nfrom pathlib import Path\nimport json\nimport os\nimport pickle\nimport re\nimport subprocess\nimport sys\nimport time\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nimport torch\n\nfrom gmic_adapter import (UPSTREAM_COMMIT, HISTORICAL_FIXTURE_COMMIT, acquire_upstream, digest_file, decode_native,\n                          infer_horizontal_flip, preprocess_array, filter_legacy_state)\nfrom mammo_reliability.audit import environment, locate_dataset, require, write_json\nfrom mammo_reliability.pipeline import initialize\nfrom mammo_reliability.training import source_fingerprint\n\nROOT = Path(\'/kaggle/working/gmic_transfer_preflight\')\nSETTINGS = {\n    \'purpose\': \'reference_reproduction_and_fit_only_technical_preflight\',\n    \'upstream_commit\': UPSTREAM_COMMIT,\n    \'models\': [1, 2, 3, 4, 5], \'weights_frozen\': True, \'precision\': \'FP32\',\n    \'upstream_source_modifications\': [],\n    \'checkpoint_compatibility\': \'Discard only unused shared_rep_filter.weight if present; strictly load active tensors.\',\n    \'reference\': \'README table with default run.sh MODEL_INDEX=1\',\n    \'reference_max_absolute_error_gate\': 0.005,\n    \'reference_gate_limit\': \'Engineering tolerance; not an accuracy test or exact old-environment replication.\',\n    \'rsna_sample\': \'Pilot fit only; 2 images per site/view/laterality plus 1 per machine, deduplicated.\',\n    \'sample_seed\': 20260924,\n    \'orientation\': \'provisional label-free outer-fifth intensity-mass heuristic; visual review required\',\n    \'rsna_preprocessing\': \'native DICOM presentation, 16-bit conversion, unmodified NYU crop/center/normalize\',\n    \'upstream_crop_erosion_iterations\': 100, \'upstream_crop_buffer_pixels\': 50,\n    \'image_size\': [2944, 1920], \'patches\': 6, \'patch_size\': [256, 256],\n    \'test_time_augmentation\': \'one deterministic center, matching released inference code\',\n    \'paper_replication\': False,\n    \'rsna_tuning_predictions\': False, \'final_test_results_opened\': False,\n    \'automatic_full_cohort_run\': False,\n    \'on_reference_mismatch\': \'Retain current-fixture failure; require historical reference agreement and current CLI parity.\',\n    \'gate_revision\': \'Historical fixture must match README; current adapter must match original current CLI inputs/outputs.\',\n    \'historical_fixture_commit\': HISTORICAL_FIXTURE_COMMIT,\n    \'current_cli_max_rounding_difference\': 0.00015,\n}\n\n\ndef load_models(upstream):\n    from src.modeling.gmic import GMIC\n    from src.constants import PERCENT_T_DICT\n    models = []\n    loading_records = []\n    for index in SETTINGS[\'models\']:\n        params = {\'device_type\': \'gpu\', \'gpu_number\': 0, \'cam_size\': (46, 30),\n                  \'K\': 6, \'crop_shape\': (256, 256), \'post_processing_dim\': 256,\n                  \'num_classes\': 2, \'use_v1_global\': False,\n                  \'percent_t\': PERCENT_T_DICT[str(index)]}\n        model = GMIC(params)\n        state = torch.load(upstream/\'models\'/f\'sample_model_{index}.p\', map_location=\'cpu\', weights_only=True)\n        active_state, ignored = filter_legacy_state(state, model.state_dict().keys())\n        model.load_state_dict(active_state, strict=True)\n        loading_records.append({\'model\': index, \'ignored_unused_keys\': ignored,\n                                \'ignored_tensor_shapes\': {k:list(state[k].shape) for k in ignored},\n                                \'active_state_loaded_strictly\': True})\n        model.eval().to(\'cuda\')\n        for parameter in model.parameters():\n            parameter.requires_grad_(False)\n        models.append(model)\n        print(\'GMIC MODEL LOADED\', index, flush=True)\n    write_json(ROOT/\'checkpoint_loading.json\', loading_records)\n    print(\'GMIC CHECKPOINT LOADING\', json.dumps(loading_records), flush=True)\n    return models\n\n\ndef infer(models, array):\n    x = torch.from_numpy(np.ascontiguousarray(array))[None, None].to(\'cuda\')\n    predictions = []\n    with torch.inference_mode():\n        for model in models:\n            out = model(x)\n            require(out.shape == (1, 2) and torch.isfinite(out).all().item(), \'Invalid model output.\')\n            require(torch.isfinite(model.saliency_map).all().item(), \'Invalid saliency map.\')\n            require(np.isfinite(model.patch_locations).all(), \'Invalid crop coordinates.\')\n            require(((out >= 0) & (out <= 1)).all().item(), \'Output outside [0,1].\')\n            predictions.append(out.cpu().numpy()[0].tolist())\n    return np.asarray(predictions)\n\n\ndef reproduce_reference(upstream, models):\n    expected = {}\n    for line in (upstream/\'README.md\').read_text().splitlines():\n        if re.match(r\'^[0-3]_[LR]-(CC|MLO)\\s*\\|\', line):\n            fields = [x.strip() for x in line.split(\'|\')]\n            expected[fields[0]] = [float(fields[1]), float(fields[2])]\n    require(len(expected) == 16, \'Reference table changed.\')\n    with (upstream/\'sample_data/exam_list_before_cropping.pkl\').open(\'rb\') as stream:\n        exams = pickle.load(stream)  # Pinned, blob-verified public author fixture.\n    records = []\n    for exam in exams:\n        for view in [\'L-CC\', \'R-CC\', \'L-MLO\', \'R-MLO\']:\n            name = exam[view][0]\n            raw = np.asarray(Image.open(upstream/\'sample_data/images\'/f\'{name}.png\'))\n            processed, _, details = preprocess_array(raw, view, exam[\'horizontal_flip\'])\n            predictions = infer(models, processed)\n            records.append({\'image\': name, \'expected_default_model\': expected[name],\n                            \'predictions_by_model\': predictions.tolist(),\n                            \'model_1_max_absolute_error\': float(np.max(np.abs(predictions[0]-expected[name]))),\n                            \'preprocessing\': details})\n            print(\'GMIC REFERENCE IMAGE\', name, records[-1][\'model_1_max_absolute_error\'], flush=True)\n    require({r[\'image\'] for r in records} == set(expected), \'Reference images missing.\')\n    max_error = max(r[\'model_1_max_absolute_error\'] for r in records)\n    result = {\'images\': len(records), \'max_absolute_error\': max_error,\n              \'tolerance\': SETTINGS[\'reference_max_absolute_error_gate\'],\n              \'passed\': max_error <= SETTINGS[\'reference_max_absolute_error_gate\'],\n              \'records\': records}\n    write_json(ROOT/\'reference_reproduction.json\', result)\n    print(\'GMIC REFERENCE RESULT\', json.dumps({k:v for k,v in result.items() if k != \'records\'}), flush=True)\n    if not result[\'passed\']:\n        parity = diagnose_original_pipeline(upstream, result)\n        historical = reproduce_historical_reference(models, expected)\n        require(parity[\'all_preprocessed_arrays_identical\'] and\n                parity[\'max_adapter_vs_cli_error\'] <= SETTINGS[\'current_cli_max_rounding_difference\'],\n                \'Adapter does not reproduce the original current pipeline.\')\n        require(historical[\'passed\'], \'Historical fixtures do not reproduce the published reference.\')\n        result[\'compatibility_passed\'] = True\n        result[\'historical_reference_max_error\'] = historical[\'max_absolute_error\']\n        result[\'current_cli_max_error\'] = parity[\'max_adapter_vs_cli_error\']\n        result[\'interpretation\'] = \'Current fixture/table mismatch retained; original fixtures and current adapter parity verified.\'\n    else:\n        result[\'compatibility_passed\'] = True\n    write_json(ROOT/\'reference_reproduction.json\', result)\n    return result\n\n\ndef diagnose_original_pipeline(upstream, reference):\n    """Compare against the author\'s complete released entry points, unchanged."""\n    destination = ROOT/\'official_cli_diagnostic\'\n    cropped = destination/\'cropped\'\n    metadata = destination/\'data.pkl\'\n    predictions = destination/\'inference\'\n    destination.mkdir(exist_ok=True)\n    env = os.environ.copy()\n    env[\'PYTHONPATH\'] = str(upstream)\n    commands = [\n        [\'src/cropping/crop_mammogram.py\', \'--input-data-folder\', str(upstream/\'sample_data/images\'),\n         \'--output-data-folder\', str(cropped), \'--exam-list-path\', str(upstream/\'sample_data/exam_list_before_cropping.pkl\'),\n         \'--cropped-exam-list-path\', str(destination/\'cropped.pkl\'), \'--num-processes\', \'2\'],\n        [\'src/optimal_centers/get_optimal_centers.py\', \'--cropped-exam-list-path\', str(destination/\'cropped.pkl\'),\n         \'--data-prefix\', str(cropped), \'--output-exam-list-path\', str(metadata), \'--num-processes\', \'2\'],\n        [\'src/scripts/run_model.py\', \'--model-path\', str(upstream/\'models\'), \'--data-path\', str(metadata),\n         \'--image-path\', str(cropped), \'--segmentation-path\', str(destination/\'no_segmentations\'),\n         \'--output-path\', str(predictions), \'--device-type\', \'gpu\', \'--gpu-number\', \'0\', \'--model-index\', \'1\'],\n    ]\n    print(\'GMIC ORIGINAL CLI DIAGNOSTIC STARTING\', flush=True)\n    for command in commands:\n        subprocess.run([sys.executable, \'-u\', *command], cwd=upstream, env=env, check=True)\n    from src.data_loading import loading\n    original = pd.read_csv(predictions/\'predictions.csv\').set_index(\'image_index\')\n    with metadata.open(\'rb\') as stream: exams = pickle.load(stream)\n    original_inputs = {}\n    for exam in exams:\n        for view in [\'L-CC\', \'R-CC\', \'L-MLO\', \'R-MLO\']:\n            name = exam[view][0]\n            image = loading.load_image(str(cropped/f\'{name}.png\'), view, exam[\'horizontal_flip\'])\n            array = loading.process_image(image, view, exam[\'best_center\'][view][0])\n            import hashlib\n            original_inputs[name] = {\'processed_sha256\': hashlib.sha256(array.tobytes()).hexdigest(),\n                                     \'best_center\': [int(x) for x in exam[\'best_center\'][view][0]]}\n    rows = []\n    for record in reference[\'records\']:\n        name = record[\'image\']\n        scores = original.loc[name, [\'benign_pred\', \'malignant_pred\']].to_numpy(float)\n        row = {\'image\': name, \'original_cli_predictions\': scores.tolist(),\n               \'adapter_predictions\': record[\'predictions_by_model\'][0],\n               \'published_predictions\': record[\'expected_default_model\'],\n               \'adapter_vs_cli_max_error\': float(np.max(np.abs(scores-record[\'predictions_by_model\'][0]))),\n               \'cli_vs_published_max_error\': float(np.max(np.abs(scores-record[\'expected_default_model\']))),\n               \'preprocessed_arrays_identical\': original_inputs[name][\'processed_sha256\'] == record[\'preprocessing\'][\'processed_sha256\'],\n               \'original_center\': original_inputs[name][\'best_center\'],\n               \'adapter_center\': record[\'preprocessing\'][\'best_center\']}\n        rows.append(row)\n        print(\'GMIC ORIGINAL CLI COMPARISON\', json.dumps(row), flush=True)\n    result = {\'status\': \'diagnosis_only_original_reference_gate_unchanged\', \'records\': rows,\n              \'all_preprocessed_arrays_identical\': all(x[\'preprocessed_arrays_identical\'] for x in rows),\n              \'max_adapter_vs_cli_error\': max(x[\'adapter_vs_cli_max_error\'] for x in rows),\n              \'max_cli_vs_published_error\': max(x[\'cli_vs_published_max_error\'] for x in rows),\n              \'rsna_inference_allowed\': False}\n    write_json(ROOT/\'original_cli_comparison.json\', result)\n    print(\'GMIC ORIGINAL CLI SUMMARY\', json.dumps({k:v for k,v in result.items() if k != \'records\'}), flush=True)\n    return result\n\n\ndef reproduce_historical_reference(models, current_expected):\n    """Use exact already-windowed author fixtures for the original README table.\n\n    Git history shows their replacement in December 2020. The initial author\n    inference code only converted these arrays to float and standardized them;\n    it explicitly disabled flipping. The checkpoint blobs are unchanged.\n    """\n    from src.data_loading.loading import standard_normalize_single_image\n    destination = ROOT/\'historical_fixtures\'\n    files = acquire_upstream(\'gmic_historical_manifest.json\', destination)\n    write_json(ROOT/\'verified_historical_files.json\', files)\n    expected = {}\n    for line in (destination/\'README.md\').read_text().splitlines():\n        if re.match(r\'^[0-3]_[LR]-(CC|MLO)\\s*\\|\', line):\n            parts = [x.strip() for x in line.split(\'|\')]\n            expected[parts[0]] = [float(parts[1]), float(parts[2])]\n    require(expected == current_expected, \'The historical and current reference tables differ.\')\n    records = []\n    for name in expected:\n        array = np.array(Image.open(destination/\'sample_data/cropped_images\'/f\'{name}.png\'), dtype=np.float32)\n        require(array.shape == (2944, 1920), \'Historical author fixture has unexpected dimensions.\')\n        standard_normalize_single_image(array)\n        predicted = infer([models[0]], array)[0]\n        record = {\'image\': name, \'expected\': expected[name], \'prediction\': predicted.tolist(),\n                  \'max_absolute_error\': float(np.max(np.abs(predicted-expected[name])))}\n        records.append(record)\n        print(\'GMIC HISTORICAL REFERENCE IMAGE\', json.dumps(record), flush=True)\n    max_error = max(x[\'max_absolute_error\'] for x in records)\n    result = {\'images\': len(records), \'fixture_commit\': HISTORICAL_FIXTURE_COMMIT,\n              \'max_absolute_error\': max_error, \'tolerance\': SETTINGS[\'reference_max_absolute_error_gate\'],\n              \'passed\': max_error <= SETTINGS[\'reference_max_absolute_error_gate\'], \'records\': records}\n    write_json(ROOT/\'historical_reference_reproduction.json\', result)\n    print(\'GMIC HISTORICAL REFERENCE SUMMARY\', json.dumps({k:v for k,v in result.items() if k != \'records\'}), flush=True)\n    return result\n\n\ndef check_rsna(models):\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    data_root = locate_dataset()\n    config = json.loads(Path(\'config.json\').read_text())\n    metadata_root = ROOT/\'cohort_identity\'\n    initialize(data_root, metadata_root, config)\n    roles_path = metadata_root/\'pilot/roles_private.csv\'\n    require(digest_file(roles_path) == \'fc0308c63300be6849c7ac9b830d09480c2b62d64df4f1d036bab12548a7e15f\',\n            \'Reserved development roles changed.\')\n    roles = pd.read_csv(roles_path)\n    require(roles.patient_id.is_unique and set(roles.cohort) == {\'pilot\'} and (roles.outer_fold == -1).all(),\n            \'Invalid reserved pilot roles.\')\n    fit_ids = set(roles.loc[roles.role.eq(\'fit\'), \'patient_id\'])\n    images = pd.read_csv(data_root/\'train.csv\')\n    fit = images.loc[images.patient_id.isin(fit_ids) & images.view.isin([\'CC\', \'MLO\'])]\n    ordered = fit.sample(frac=1, random_state=SETTINGS[\'sample_seed\'])\n    sample = pd.concat([ordered.groupby([\'site_id\', \'view\', \'laterality\'], sort=True).head(2),\n                        ordered.groupby(\'machine_id\', sort=True).head(1)]).drop_duplicates(\'image_id\')\n    require(set(sample.patient_id).issubset(fit_ids), \'Engineering sample includes nonfit patients.\')\n    sample.to_csv(ROOT/\'engineering_sample_private.csv\', index=False)\n    records, thumbnails = [], []\n    for count, row in enumerate(sample.itertuples(index=False), 1):\n        started = time.monotonic()\n        raw, header = decode_native(data_root/\'train_images\'/str(row.patient_id)/f\'{row.image_id}.dcm\')\n        view = f\'{row.laterality}-{row.view}\'\n        flip, orientation = infer_horizontal_flip(raw, view)\n        processed, window, details = preprocess_array(raw, view, flip)\n        predictions = infer(models, processed)\n        record = {\'sample\': count, \'site_id\': int(row.site_id), \'machine_id\': int(row.machine_id),\n                  \'view\': view, **header, **orientation, **details,\n                  \'elapsed_seconds\': time.monotonic()-started}\n        records.append(record)\n        # No RSNA scores or cancer labels in aggregate logs or contact-sheet titles.\n        np.save(ROOT/f\'sample_{count:02d}_predictions_private.npy\', predictions)\n        thumbnails.append((np.asarray(Image.fromarray(raw).resize((300, 460))),\n                           np.asarray(Image.fromarray(window).resize((300, 460)))))\n        print(\'GMIC RSNA TECHNICAL IMAGE\', json.dumps(record), flush=True)\n    for start in range(0, len(records), 4):\n        fig, axes = plt.subplots(4, 2, figsize=(8, 16))\n        for offset, pair in enumerate(axes):\n            idx = start+offset\n            for ax in pair: ax.axis(\'off\')\n            if idx >= len(records): continue\n            item = records[idx]\n            for col, image in enumerate(thumbnails[idx]):\n                pair[col].imshow(image, cmap=\'gray\')\n                pair[col].set_title(f"Sample {idx+1} site {item[\'site_id\']} {item[\'view\']} | "\n                                    + (\'native presentation\' if col == 0 else \'GMIC window\'))\n        fig.suptitle(\'Private technical orientation/crop review; no diagnostic interpretation\')\n        fig.tight_layout()\n        fig.savefig(ROOT/f\'contact_sheet_{start//4+1:02d}_private.png\', dpi=100)\n        plt.close(fig)\n    result = {\'images\': len(records), \'sampling_partition\': \'pilot_fit\', \'records\': records,\n              \'visual_review_status\': \'pending\', \'full_cohort_transfer_evaluation_authorized_by_gate\': False}\n    write_json(ROOT/\'rsna_technical_summary.json\', result)\n    return result\n\n\ndef main():\n    ROOT.mkdir(parents=True, exist_ok=True)\n    require(torch.cuda.is_available(), \'Kaggle GPU required.\')\n    require(source_fingerprint() == \'f4178761b59233ffe1048218d234624c22764c8129d11c48fe30821cbe079a7c\',\n            \'Original model package changed.\')\n    torch.manual_seed(17)\n    np.random.seed(17)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.allow_tf32 = False\n    declared = {**SETTINGS, \'environment\': environment(),\n                \'script_sha256\': digest_file(__file__),\n                \'adapter_sha256\': digest_file(\'gmic_adapter.py\'),\n                \'manifest_sha256\': digest_file(\'gmic_upstream_manifest.json\')}\n    write_json(ROOT/\'declared_plan.json\', declared)\n    print(\'GMIC DECLARED PLAN\', json.dumps(declared), flush=True)\n    upstream = ROOT/\'upstream\'\n    files = acquire_upstream(\'gmic_upstream_manifest.json\', upstream)\n    write_json(ROOT/\'verified_upstream_files.json\', files)\n    sys.path.insert(0, str(upstream))\n    models = load_models(upstream)\n    reference = reproduce_reference(upstream, models)\n    technical = check_rsna(models)\n    result = {\'status\': \'numeric_preflight_completed_visual_review_pending\',\n              \'reference_compatibility_passed\': reference[\'compatibility_passed\'],\n              \'current_fixture_vs_readme_passed\': reference[\'passed\'],\n              \'current_fixture_vs_readme_max_error\': reference[\'max_absolute_error\'],\n              \'historical_reference_max_error\': reference.get(\'historical_reference_max_error\'),\n              \'current_cli_max_error\': reference.get(\'current_cli_max_error\'),\n              \'rsna_fit_images_checked\': technical[\'images\'], \'final_test_results_opened\': False,\n              \'rsna_accuracy_estimated\': False, \'visual_review_required\': True,\n              \'upstream_commit\': UPSTREAM_COMMIT,\n              \'peak_gpu_allocated_gib\': torch.cuda.max_memory_allocated()/1024**3}\n    write_json(ROOT/\'completed.json\', result)\n    print(\'GMIC PREFLIGHT COMPLETED\', json.dumps(result), flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n')
target = PROJECT / 'gmic_fullfield.py'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text('"""RSNA full-field adaptation for fixed GMIC weights; not NYU preprocessing."""\nimport numpy as np\nfrom PIL import Image\nfrom gmic_adapter import infer_horizontal_flip\n\n\ndef fullfield_array(raw, view, height=2944, width=1920):\n    """Orient, fit the complete image extent, and pad without a crop window.\n\n    Larger images are downsampled with preserved aspect ratio; smaller images\n    are not enlarged. This avoids geometric exclusion but cannot guarantee\n    preservation of small diagnostic details. No label enters this function.\n    """\n    if min(height, width) <= 0:\n        raise ValueError(\'Invalid target dimensions.\')\n    flip_flag, orientation = infer_horizontal_flip(raw, view)\n    flip = ((flip_flag == \'NO\' and view.startswith(\'R-\')) or\n            (flip_flag == \'YES\' and view.startswith(\'L-\')))\n    oriented = np.fliplr(raw) if flip else raw\n    h, w = oriented.shape\n    scale = min(1., height/h, width/w)\n    new_h, new_w = max(1, round(h*scale)), max(1, round(w*scale))\n    resized = np.asarray(Image.fromarray(oriented.astype(np.float32)).resize(\n        (new_w, new_h), resample=Image.Resampling.BILINEAR))\n    canvas = np.zeros((height, width), dtype=np.float32)\n    # Chest wall on the left. Match the released CC/MLO padding convention.\n    top = (height-new_h)//2 if view.endswith(\'CC\') else 0\n    canvas[top:top+new_h, :new_w] = resized\n    processed = canvas.copy()\n    processed -= processed.mean()\n    processed /= max(float(processed.std()), 1e-5)\n    if not np.isfinite(processed).all() or processed.std() < .99:\n        raise ValueError(\'Invalid standardized full-field input.\')\n    details = {**orientation, \'horizontal_flip\': flip_flag, \'actual_flip_applied\': flip,\n               \'raw_shape\': [h,w], \'resized_shape\': [new_h,new_w],\n               \'canvas_shape\': [height,width], \'scale\': float(scale),\n               \'padding_top\': top, \'padding_left\': 0,\n               \'source_window\': [0,h,0,w], \'geometric_cropping\': False,\n               \'normalized_mean\': float(processed.mean()), \'normalized_std\': float(processed.std())}\n    return processed, canvas, details\n')
target = PROJECT / 'gmic_fullfield_preflight.py'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text('"""Audit full-field GMIC adaptation on the exact original 20 fit images."""\nfrom pathlib import Path\nimport json\nimport sys\nimport time\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom PIL import Image\n\nfrom gmic_adapter import digest_file, decode_native, filter_legacy_state, UPSTREAM_COMMIT\nfrom gmic_fullfield import fullfield_array\nfrom gmic_preflight import infer\nfrom mammo_reliability.audit import locate_dataset, require, write_json, environment\n\nOUTPUT = Path(\'/kaggle/working/gmic_fullfield_preflight\')\nEXPECTED_PLAN = \'6f0aff552657aebfe4a4da73ddeacfdf29e62daaa3b590c34e7ef499949fdfc2\'\n\n\ndef verified_preflight():\n    candidates = [Path(\'/kaggle/input/notebooks/aseqwss/09-rsna-gmic-transfer-preflight/gmic_transfer_preflight\'),\n                  Path(\'/kaggle/input/09-rsna-gmic-transfer-preflight/gmic_transfer_preflight\')]\n    existing = [x for x in candidates if x.is_dir()]\n    require(len(existing) == 1, \'Attach private notebook 09 version 4.\')\n    base = existing[0]\n    require(digest_file(base/\'declared_plan.json\') == EXPECTED_PLAN, \'Wrong preflight version.\')\n    result = json.loads((base/\'completed.json\').read_text())\n    require(result[\'reference_compatibility_passed\'], \'Compatibility checks did not pass.\')\n    require(result[\'rsna_fit_images_checked\'] == 20 and not result[\'final_test_results_opened\'], \'Unexpected preflight.\')\n    manifest = json.loads(Path(\'gmic_upstream_manifest.json\').read_text())\n    require(manifest[\'commit\'] == UPSTREAM_COMMIT, \'Changed upstream source.\')\n    upstream = base/\'upstream\'\n    for name, blob in manifest[\'files\'].items():\n        require(digest_file(upstream/name, \'sha1\', git_blob=True) == blob, f\'Upstream file changed: {name}\')\n    return base, upstream\n\n\ndef load_fixed_models(upstream, output):\n    sys.path.insert(0, str(upstream))\n    from src.modeling.gmic import GMIC\n    from src.constants import PERCENT_T_DICT\n    models, records = [], []\n    for index in range(1,6):\n        params = {\'device_type\':\'gpu\',\'gpu_number\':0,\'cam_size\':(46,30),\'K\':6,\n                  \'crop_shape\':(256,256),\'post_processing_dim\':256,\'num_classes\':2,\n                  \'use_v1_global\':False,\'percent_t\':PERCENT_T_DICT[str(index)]}\n        model = GMIC(params)\n        checkpoint = upstream/\'models\'/f\'sample_model_{index}.p\'\n        state = torch.load(checkpoint, map_location=\'cpu\', weights_only=True)\n        active, ignored = filter_legacy_state(state, model.state_dict().keys())\n        model.load_state_dict(active, strict=True)\n        model.eval().to(\'cuda\')\n        for parameter in model.parameters(): parameter.requires_grad_(False)\n        models.append(model)\n        records.append({\'model\':index,\'checkpoint_sha256\':digest_file(checkpoint),\'ignored_unused_keys\':ignored})\n    write_json(output/\'fixed_models.json\',records)\n    return models\n\n\ndef main():\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    OUTPUT.mkdir(parents=True,exist_ok=True)\n    require(torch.cuda.is_available(), \'GPU required.\')\n    torch.backends.cudnn.benchmark=False\n    torch.backends.cudnn.deterministic=True\n    torch.backends.cuda.matmul.allow_tf32=False\n    torch.backends.cudnn.allow_tf32=False\n    base, upstream = verified_preflight()\n    roles_path=base/\'cohort_identity/pilot/roles_private.csv\'\n    require(digest_file(roles_path)==\'fc0308c63300be6849c7ac9b830d09480c2b62d64df4f1d036bab12548a7e15f\',\'Roles changed.\')\n    roles=pd.read_csv(roles_path)\n    fit_ids=set(roles.loc[roles.role.eq(\'fit\'),\'patient_id\'])\n    sample=pd.read_csv(base/\'engineering_sample_private.csv\')\n    require(len(sample)==20 and sample.image_id.is_unique and set(sample.patient_id).issubset(fit_ids), \'Wrong engineering sample.\')\n    declaration={\'purpose\':\'full_field_geometry_repair_on_same_20_fit_images\',\'preflight_plan_sha256\':EXPECTED_PLAN,\n                 \'preprocessing\':\'full original field, horizontal orientation heuristic, downscale only to fit, left-aligned padding\',\n                 \'target_shape\':[2944,1920],\'interpolation\':\'bilinear\',\'image_extent_cropped\':False,\n                 \'caution\':\'Downsampling can lose fine detail; this is an explicit RSNA adaptation, not exact NYU preprocessing.\',\n                 \'frozen_models\':5,\'accuracy_calculated\':False,\'final_test_results_opened\':False,\n                 \'environment\':environment(),\'script_sha256\':digest_file(__file__),\n                 \'fullfield_sha256\':digest_file(\'gmic_fullfield.py\'),\'adapter_sha256\':digest_file(\'gmic_adapter.py\')}\n    write_json(OUTPUT/\'declared_plan.json\',declaration)\n    print(\'FULLFIELD DECLARED PLAN\',json.dumps(declaration),flush=True)\n    models=load_fixed_models(upstream,OUTPUT)\n    data_root=locate_dataset()\n    require(digest_file(data_root/\'train.csv\')==\'35408731b67b87272f85874ee3c4c9d5cea7a2c4fbcaeac3fbe415ba1a5817c1\',\'Metadata changed.\')\n    records,thumbs=[],[]\n    for i,row in enumerate(sample.itertuples(index=False),1):\n        started=time.monotonic()\n        raw,header=decode_native(data_root/\'train_images\'/str(row.patient_id)/f\'{row.image_id}.dcm\')\n        view=f\'{row.laterality}-{row.view}\'\n        array,canvas,geometry=fullfield_array(raw,view)\n        predictions=infer(models,array)\n        np.save(OUTPUT/f\'sample_{i:02d}_predictions_private.npy\',predictions)\n        records.append({\'sample\':i,\'site_id\':int(row.site_id),\'machine_id\':int(row.machine_id),\n                        \'view\':view,**header,**geometry,\'seconds\':time.monotonic()-started})\n        pair=[]\n        for img in [raw,canvas]:\n            # Pillow cannot reduce every uint16 mode; float preserves display range.\n            thumb=Image.fromarray(img.astype(np.float32));thumb.thumbnail((500,500));pair.append(np.asarray(thumb))\n        thumbs.append(pair)\n        print(\'FULLFIELD IMAGE\',json.dumps(records[-1]),flush=True)\n    for start in range(0,20,4):\n        fig,axes=plt.subplots(4,2,figsize=(10,16),layout=\'constrained\')\n        for j in range(4):\n            i=start+j\n            for col in range(2):\n                axes[j,col].imshow(thumbs[i][col],cmap=\'gray\',aspect=\'equal\')\n                axes[j,col].axis(\'off\')\n                axes[j,col].set_title(f"Sample {i+1}, site {records[i][\'site_id\']}, {records[i][\'view\']}\\n"\n                                      +(\'Original presentation\' if col==0 else \'Full-field model input\'),fontsize=10)\n        fig.suptitle(\'Private engineering review; no lesion or cancer interpretation\')\n        fig.savefig(OUTPUT/f\'contact_sheet_{start//4+1:02d}_private.png\',dpi=100)\n        plt.close(fig)\n    write_json(OUTPUT/\'technical_images.json\',records)\n    result={\'status\':\'numeric_fullfield_preflight_completed_visual_review_pending\',\'images\':20,\n            \'final_test_results_opened\':False,\'accuracy_calculated\':False,\'visual_review_required\':True,\n            \'minimum_scale\':min(r[\'scale\'] for r in records),\'maximum_scale\':max(r[\'scale\'] for r in records),\n            \'peak_gpu_allocated_gib\':torch.cuda.max_memory_allocated()/1024**3,\n            \'declared_plan_sha256\':digest_file(OUTPUT/\'declared_plan.json\')}\n    write_json(OUTPUT/\'completed.json\',result)\n    print(\'FULLFIELD COMPLETED\',json.dumps(result),flush=True)\n\n\nif __name__==\'__main__\': main()\n')
target = PROJECT / 'gmic_visual_review.json'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text('{\n  "observed_date": "2026-09-24",\n  "notebook": "10_rsna_gmic_fullfield_check",\n  "version": 2,\n  "script_version_id": 352360882,\n  "declared_plan_sha256": "94c490d11b274dedc01998942e9b0251ef73fbcdffdcb6e8d8d502174b92d81e",\n  "reviewer": "AI assistant, technical visual inspection; no radiologist review",\n  "engineering_visual_review_passed": true,\n  "images_reviewed": 20,\n  "sample_partition": "same deterministic pilot fitting images used by notebook 09",\n  "contact_sheets_reviewed": [\n    "contact_sheet_01_private.png",\n    "contact_sheet_02_private.png",\n    "contact_sheet_03_private.png",\n    "contact_sheet_04_private.png",\n    "contact_sheet_05_private.png"\n  ],\n  "observations": [\n    "All 20 original/model-input pairs inspected at contact-sheet scale.",\n    "No visible full-image boundary exclusion, blank output, or gross grayscale inversion in these pairs.",\n    "Model inputs consistently present the chest wall on the left in these examples.",\n    "Sample 11 now retains the visible breast; the original fixed window was largely background.",\n    "Substantial background and small breast areas remain in some acquisitions, including samples 5 and 16."\n  ],\n  "limitations": [\n    "This is an engineering review of 20 images, not diagnostic or lesion-preservation validation.",\n    "Downscaling ranged from 0.3906408952187182 to 1; fine features can be lost.",\n    "Orientation is a heuristic; success on these examples does not validate the entire cohort.",\n    "No cancer predictions were judged and no RSNA cohort accuracy was used to select the repair."\n  ],\n  "allowed_next_stage": "reserved pilot transfer feasibility only",\n  "final_evaluation_unlocked": false\n}\n')
REFERENCE = {'raw_threshold': 0.029317490197718, 'classification': [{'role': 'fit', 'site': 'pooled', 'n_breasts': 3574, 'n_positive': 73, 'tp': 66, 'fn': 7, 'tn': 1819, 'fp': 1682, 'sensitivity': 0.9041095890410958, 'specificity': 0.5195658383319052, 'ppv_in_study_sample': 0.03775743707093822, 'npv_in_study_sample': 0.9961664841182913, 'roc_auc': 0.8239446263885465, 'average_precision': 0.22096024800465539, 'brier': 0.018082507933630296, 'log_loss': 0.08188265113647725, 'ece_10_equal_width_bins': 0.0019724597215308847, 'threshold': 0.00723728353840227}, {'role': 'fit', 'site': 1, 'n_breasts': 1746, 'n_positive': 38, 'tp': 32, 'fn': 6, 'tn': 1035, 'fp': 673, 'sensitivity': 0.8421052631578947, 'specificity': 0.6059718969555035, 'ppv_in_study_sample': 0.04539007092198582, 'npv_in_study_sample': 0.9942363112391931, 'roc_auc': 0.8447399235794404, 'average_precision': 0.26208547926735204, 'brier': 0.019448109526458212, 'log_loss': 0.08563974663824836, 'ece_10_equal_width_bins': 0.010312709053004805, 'threshold': 0.00723728353840227}, {'role': 'fit', 'site': 2, 'n_breasts': 1828, 'n_positive': 35, 'tp': 34, 'fn': 1, 'tn': 784, 'fp': 1009, 'sensitivity': 0.9714285714285714, 'specificity': 0.4372559955382041, 'ppv_in_study_sample': 0.032598274209012464, 'npv_in_study_sample': 0.9987261146496815, 'roc_auc': 0.8279499641462832, 'average_precision': 0.25190037877890453, 'brier': 0.016778164180305602, 'log_loss': 0.07829409055327573, 'ece_10_equal_width_bins': 0.011063672837727575, 'threshold': 0.00723728353840227}, {'role': 'tune', 'site': 'pooled', 'n_breasts': 1192, 'n_positive': 26, 'tp': 18, 'fn': 8, 'tn': 661, 'fp': 505, 'sensitivity': 0.6923076923076923, 'specificity': 0.5668953687821612, 'ppv_in_study_sample': 0.03441682600382409, 'npv_in_study_sample': 0.9880418535127056, 'roc_auc': 0.7261182214012402, 'average_precision': 0.14682941906524094, 'brier': 0.020324179516721386, 'log_loss': 0.09678423113159917, 'ece_10_equal_width_bins': 0.0048126079744467126, 'threshold': 0.00723728353840227}, {'role': 'tune', 'site': 1, 'n_breasts': 582, 'n_positive': 13, 'tp': 9, 'fn': 4, 'tn': 379, 'fp': 190, 'sensitivity': 0.6923076923076923, 'specificity': 0.6660808435852372, 'ppv_in_study_sample': 0.04522613065326633, 'npv_in_study_sample': 0.9895561357702349, 'roc_auc': 0.760848992834933, 'average_precision': 0.44247011267231967, 'brier': 0.0193364269773492, 'log_loss': 0.08582981001868896, 'ece_10_equal_width_bins': 0.011699866927426035, 'threshold': 0.00723728353840227}, {'role': 'tune', 'site': 2, 'n_breasts': 610, 'n_positive': 13, 'tp': 9, 'fn': 4, 'tn': 282, 'fp': 315, 'sensitivity': 0.6923076923076923, 'specificity': 0.4723618090452261, 'ppv_in_study_sample': 0.027777777777777776, 'npv_in_study_sample': 0.986013986013986, 'roc_auc': 0.653395181033372, 'average_precision': 0.0433427005108422, 'brier': 0.02126659259526993, 'log_loss': 0.10723582635735937, 'ece_10_equal_width_bins': 0.006238737839900565, 'threshold': 0.00723728353840227}], 'review': [{'method': 'score_confidence', 'budget': 0.01, 'negative_breasts': 669, 'reviewed_breasts': 6, 'false_negatives': 8, 'false_negatives_flagged': 0, 'fn_capture_fraction': 0.0, 'review_yield': 0.0, 'actual_review_fraction': 0.008968609865470852, 'random_expected_fn_capture': 0.008968609865470852, 'interpretation': 'flagging potential review candidates; no human review or cancer recovery was measured', 'review_fraction_all_tune_breasts': 0.0050335570469798654}, {'method': 'score_disagreement', 'budget': 0.01, 'negative_breasts': 669, 'reviewed_breasts': 6, 'false_negatives': 8, 'false_negatives_flagged': 0, 'fn_capture_fraction': 0.0, 'review_yield': 0.0, 'actual_review_fraction': 0.008968609865470852, 'random_expected_fn_capture': 0.008968609865470852, 'interpretation': 'flagging potential review candidates; no human review or cancer recovery was measured', 'review_fraction_all_tune_breasts': 0.0050335570469798654}, {'method': 'score_uncertainty', 'budget': 0.01, 'negative_breasts': 669, 'reviewed_breasts': 6, 'false_negatives': 8, 'false_negatives_flagged': 0, 'fn_capture_fraction': 0.0, 'review_yield': 0.0, 'actual_review_fraction': 0.008968609865470852, 'random_expected_fn_capture': 0.008968609865470852, 'interpretation': 'flagging potential review candidates; no human review or cancer recovery was measured', 'review_fraction_all_tune_breasts': 0.0050335570469798654}, {'method': 'score_confidence_disagreement', 'budget': 0.01, 'negative_breasts': 669, 'reviewed_breasts': 6, 'false_negatives': 8, 'false_negatives_flagged': 0, 'fn_capture_fraction': 0.0, 'review_yield': 0.0, 'actual_review_fraction': 0.008968609865470852, 'random_expected_fn_capture': 0.008968609865470852, 'interpretation': 'flagging potential review candidates; no human review or cancer recovery was measured', 'review_fraction_all_tune_breasts': 0.0050335570469798654}, {'method': 'score_confidence_uncertainty', 'budget': 0.01, 'negative_breasts': 669, 'reviewed_breasts': 6, 'false_negatives': 8, 'false_negatives_flagged': 0, 'fn_capture_fraction': 0.0, 'review_yield': 0.0, 'actual_review_fraction': 0.008968609865470852, 'random_expected_fn_capture': 0.008968609865470852, 'interpretation': 'flagging potential review candidates; no human review or cancer recovery was measured', 'review_fraction_all_tune_breasts': 0.0050335570469798654}, {'method': 'score_combined', 'budget': 0.01, 'negative_breasts': 669, 'reviewed_breasts': 6, 'false_negatives': 8, 'false_negatives_flagged': 0, 'fn_capture_fraction': 0.0, 'review_yield': 0.0, 'actual_review_fraction': 0.008968609865470852, 'random_expected_fn_capture': 0.008968609865470852, 'interpretation': 'flagging potential review candidates; no human review or cancer recovery was measured', 'review_fraction_all_tune_breasts': 0.0050335570469798654}, {'method': 'score_confidence', 'budget': 0.05, 'negative_breasts': 669, 'reviewed_breasts': 33, 'false_negatives': 8, 'false_negatives_flagged': 0, 'fn_capture_fraction': 0.0, 'review_yield': 0.0, 'actual_review_fraction': 0.04932735426008968, 'random_expected_fn_capture': 0.04932735426008968, 'interpretation': 'flagging potential review candidates; no human review or cancer recovery was measured', 'review_fraction_all_tune_breasts': 0.027684563758389263}, {'method': 'score_disagreement', 'budget': 0.05, 'negative_breasts': 669, 'reviewed_breasts': 33, 'false_negatives': 8, 'false_negatives_flagged': 1, 'fn_capture_fraction': 0.125, 'review_yield': 0.030303030303030304, 'actual_review_fraction': 0.04932735426008968, 'random_expected_fn_capture': 0.04932735426008968, 'interpretation': 'flagging potential review candidates; no human review or cancer recovery was measured', 'review_fraction_all_tune_breasts': 0.027684563758389263}, {'method': 'score_uncertainty', 'budget': 0.05, 'negative_breasts': 669, 'reviewed_breasts': 33, 'false_negatives': 8, 'false_negatives_flagged': 0, 'fn_capture_fraction': 0.0, 'review_yield': 0.0, 'actual_review_fraction': 0.04932735426008968, 'random_expected_fn_capture': 0.04932735426008968, 'interpretation': 'flagging potential review candidates; no human review or cancer recovery was measured', 'review_fraction_all_tune_breasts': 0.027684563758389263}, {'method': 'score_confidence_disagreement', 'budget': 0.05, 'negative_breasts': 669, 'reviewed_breasts': 33, 'false_negatives': 8, 'false_negatives_flagged': 1, 'fn_capture_fraction': 0.125, 'review_yield': 0.030303030303030304, 'actual_review_fraction': 0.04932735426008968, 'random_expected_fn_capture': 0.04932735426008968, 'interpretation': 'flagging potential review candidates; no human review or cancer recovery was measured', 'review_fraction_all_tune_breasts': 0.027684563758389263}, {'method': 'score_confidence_uncertainty', 'budget': 0.05, 'negative_breasts': 669, 'reviewed_breasts': 33, 'false_negatives': 8, 'false_negatives_flagged': 0, 'fn_capture_fraction': 0.0, 'review_yield': 0.0, 'actual_review_fraction': 0.04932735426008968, 'random_expected_fn_capture': 0.04932735426008968, 'interpretation': 'flagging potential review candidates; no human review or cancer recovery was measured', 'review_fraction_all_tune_breasts': 0.027684563758389263}, {'method': 'score_combined', 'budget': 0.05, 'negative_breasts': 669, 'reviewed_breasts': 33, 'false_negatives': 8, 'false_negatives_flagged': 1, 'fn_capture_fraction': 0.125, 'review_yield': 0.030303030303030304, 'actual_review_fraction': 0.04932735426008968, 'random_expected_fn_capture': 0.04932735426008968, 'interpretation': 'flagging potential review candidates; no human review or cancer recovery was measured', 'review_fraction_all_tune_breasts': 0.027684563758389263}, {'method': 'score_confidence', 'budget': 0.1, 'negative_breasts': 669, 'reviewed_breasts': 66, 'false_negatives': 8, 'false_negatives_flagged': 0, 'fn_capture_fraction': 0.0, 'review_yield': 0.0, 'actual_review_fraction': 0.09865470852017937, 'random_expected_fn_capture': 0.09865470852017937, 'interpretation': 'flagging potential review candidates; no human review or cancer recovery was measured', 'review_fraction_all_tune_breasts': 0.05536912751677853}, {'method': 'score_disagreement', 'budget': 0.1, 'negative_breasts': 669, 'reviewed_breasts': 66, 'false_negatives': 8, 'false_negatives_flagged': 3, 'fn_capture_fraction': 0.375, 'review_yield': 0.045454545454545456, 'actual_review_fraction': 0.09865470852017937, 'random_expected_fn_capture': 0.09865470852017937, 'interpretation': 'flagging potential review candidates; no human review or cancer recovery was measured', 'review_fraction_all_tune_breasts': 0.05536912751677853}, {'method': 'score_uncertainty', 'budget': 0.1, 'negative_breasts': 669, 'reviewed_breasts': 66, 'false_negatives': 8, 'false_negatives_flagged': 2, 'fn_capture_fraction': 0.25, 'review_yield': 0.030303030303030304, 'actual_review_fraction': 0.09865470852017937, 'random_expected_fn_capture': 0.09865470852017937, 'interpretation': 'flagging potential review candidates; no human review or cancer recovery was measured', 'review_fraction_all_tune_breasts': 0.05536912751677853}, {'method': 'score_confidence_disagreement', 'budget': 0.1, 'negative_breasts': 669, 'reviewed_breasts': 66, 'false_negatives': 8, 'false_negatives_flagged': 3, 'fn_capture_fraction': 0.375, 'review_yield': 0.045454545454545456, 'actual_review_fraction': 0.09865470852017937, 'random_expected_fn_capture': 0.09865470852017937, 'interpretation': 'flagging potential review candidates; no human review or cancer recovery was measured', 'review_fraction_all_tune_breasts': 0.05536912751677853}, {'method': 'score_confidence_uncertainty', 'budget': 0.1, 'negative_breasts': 669, 'reviewed_breasts': 66, 'false_negatives': 8, 'false_negatives_flagged': 2, 'fn_capture_fraction': 0.25, 'review_yield': 0.030303030303030304, 'actual_review_fraction': 0.09865470852017937, 'random_expected_fn_capture': 0.09865470852017937, 'interpretation': 'flagging potential review candidates; no human review or cancer recovery was measured', 'review_fraction_all_tune_breasts': 0.05536912751677853}, {'method': 'score_combined', 'budget': 0.1, 'negative_breasts': 669, 'reviewed_breasts': 66, 'false_negatives': 8, 'false_negatives_flagged': 2, 'fn_capture_fraction': 0.25, 'review_yield': 0.030303030303030304, 'actual_review_fraction': 0.09865470852017937, 'random_expected_fn_capture': 0.09865470852017937, 'interpretation': 'flagging potential review candidates; no human review or cancer recovery was measured', 'review_fraction_all_tune_breasts': 0.05536912751677853}, {'method': 'score_confidence', 'budget': 0.2, 'negative_breasts': 669, 'reviewed_breasts': 133, 'false_negatives': 8, 'false_negatives_flagged': 5, 'fn_capture_fraction': 0.625, 'review_yield': 0.03759398496240601, 'actual_review_fraction': 0.19880418535127056, 'random_expected_fn_capture': 0.19880418535127056, 'interpretation': 'flagging potential review candidates; no human review or cancer recovery was measured', 'review_fraction_all_tune_breasts': 0.1115771812080537}, {'method': 'score_disagreement', 'budget': 0.2, 'negative_breasts': 669, 'reviewed_breasts': 133, 'false_negatives': 8, 'false_negatives_flagged': 3, 'fn_capture_fraction': 0.375, 'review_yield': 0.022556390977443608, 'actual_review_fraction': 0.19880418535127056, 'random_expected_fn_capture': 0.19880418535127056, 'interpretation': 'flagging potential review candidates; no human review or cancer recovery was measured', 'review_fraction_all_tune_breasts': 0.1115771812080537}, {'method': 'score_uncertainty', 'budget': 0.2, 'negative_breasts': 669, 'reviewed_breasts': 133, 'false_negatives': 8, 'false_negatives_flagged': 2, 'fn_capture_fraction': 0.25, 'review_yield': 0.015037593984962405, 'actual_review_fraction': 0.19880418535127056, 'random_expected_fn_capture': 0.19880418535127056, 'interpretation': 'flagging potential review candidates; no human review or cancer recovery was measured', 'review_fraction_all_tune_breasts': 0.1115771812080537}, {'method': 'score_confidence_disagreement', 'budget': 0.2, 'negative_breasts': 669, 'reviewed_breasts': 133, 'false_negatives': 8, 'false_negatives_flagged': 4, 'fn_capture_fraction': 0.5, 'review_yield': 0.03007518796992481, 'actual_review_fraction': 0.19880418535127056, 'random_expected_fn_capture': 0.19880418535127056, 'interpretation': 'flagging potential review candidates; no human review or cancer recovery was measured', 'review_fraction_all_tune_breasts': 0.1115771812080537}, {'method': 'score_confidence_uncertainty', 'budget': 0.2, 'negative_breasts': 669, 'reviewed_breasts': 133, 'false_negatives': 8, 'false_negatives_flagged': 3, 'fn_capture_fraction': 0.375, 'review_yield': 0.022556390977443608, 'actual_review_fraction': 0.19880418535127056, 'random_expected_fn_capture': 0.19880418535127056, 'interpretation': 'flagging potential review candidates; no human review or cancer recovery was measured', 'review_fraction_all_tune_breasts': 0.1115771812080537}, {'method': 'score_combined', 'budget': 0.2, 'negative_breasts': 669, 'reviewed_breasts': 133, 'false_negatives': 8, 'false_negatives_flagged': 3, 'fn_capture_fraction': 0.375, 'review_yield': 0.022556390977443608, 'actual_review_fraction': 0.19880418535127056, 'random_expected_fn_capture': 0.19880418535127056, 'interpretation': 'flagging potential review candidates; no human review or cancer recovery was measured', 'review_fraction_all_tune_breasts': 0.1115771812080537}]}

roots = [Path('/kaggle/input/notebooks/aseqwss/11-rsna-gmic-reserved-pilot/gmic_reserved_pilot'),
         Path('/kaggle/input/11-rsna-gmic-reserved-pilot/gmic_reserved_pilot')]
existing = [p for p in roots if (p/'completed.json').is_file()]
assert len(existing) == 1, 'Attach private notebook 11 version 2 output.'
SOURCE = existing[0]
OUT = Path('/kaggle/working/gmic_operating_point')
subprocess.run([sys.executable, '-u', 'gmic_operating_point.py', '--input', str(SOURCE),
                '--output', str(OUT), '--private-output', '/kaggle/working/gmic_analysis_private'],
               cwd=PROJECT, check=True)
result = json.loads((OUT/'results.json').read_text())
assert abs(result['raw_threshold']-REFERENCE['raw_threshold']) < 1e-12
for actual, expected in zip(result['classification'], REFERENCE['classification'], strict=True):
    for key in ['role', 'site', 'tp', 'fn', 'tn', 'fp']:
        assert actual[key] == expected[key], ('Confusion counts changed', key)
for actual, expected in zip(result['review'], REFERENCE['review'], strict=True):
    for key in ['method', 'budget', 'false_negatives', 'reviewed_breasts', 'false_negatives_flagged']:
        assert actual[key] == expected[key], ('Review selection result changed', key)
print('CPU_ANALYSIS_REPRODUCED', json.dumps({'classification_groups': 6, 'review_comparisons': 24,
        'final_test_results_opened': False}))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pydicom>=3,<4',
                'pylibjpeg>=2,<3', 'pylibjpeg-openjpeg>=2,<3', 'pylibjpeg-libjpeg>=2.1,<3'], check=True)
# Fresh process is essential: pydicom discovers decoder plugins at import time,
# before the optional installation above. Do not reuse a stale decoder registry.
subprocess.run([sys.executable, '-c',
                'import sys; from gmic_orientation_review import run; run(sys.argv[1], sys.argv[2])',
                str(SOURCE), '/kaggle/working/gmic_orientation_review'], cwd=PROJECT, check=True)
print('DEVELOPMENT_ANALYSIS_COMPLETE', json.dumps({'analysis_reproduced': True,
        'orientation_display_prepared': True, 'visual_review_pending': True,
        'final_test_results_opened': False}))
